# Spendly V6 — Kaggle T4: faster features + plateau-aware LSTM

**Use R3 data unchanged.** All 600 training users, 100 validation users and 100 separate
calibration users. Architecture stays LSTM forecasting, Isolation Forest experiments
and rule-based recommendations. This is a training experiment, not a proven model improvement.

## What V6 changes
- Optimized array-based recurrence/weekly feature construction; same 26 feature values
  as V5, checked at tight tolerance. Cached independent users run in CPU-only worker
  processes (2 by default, max4). No TensorFlow or CUDA is loaded by feature workers.
- **LSTM uses GPU0 or fails immediately**. No CPU benchmark/fallback. Kaggle T4x2
  supplies two GPUs, but V6 intentionally uses **one T4** to avoid small-model
  synchronization overhead and retain Adam clipnorm=1 semantics. GPU1 may be idle.
- Three candidates: 32-unit fixed learning rate control, 32-unit plateau reduction,
  64-unit plateau reduction. LR starts .0003; plateau patience4 halves it to floor1e-5.
  Early stopping patience16, cap120. The exact selected LR history is replayed at refit.
- Log weighted MAE separately from regularization and save the refit schedule and
  timing report. No arbitrary extra epochs, altered labels or test-based selection.

## Setup in Kaggle
1. Save/download your existing V5 reports before changing sessions.
2. Import `Spending_Model_Upload_V6_R3_Kaggle.ipynb`: **File > Import Notebook**.
3. **Settings / Session options > Accelerator > GPU T4 x2** (or one GPU if offered).
   Reconnect if prompted. GPU quota/account verification is controlled by Kaggle.
4. Attach the private R3 development dataset via **Add Input / Add Data** containing
   `manifest.json`, `train.zip`, `validation.zip`, `calibration.zip`. CSVs work too.
   Base Windows folder: `D:\ICS\YEAR 4\sem1\IS\model related\datasets\spendly_synthetic_r3\`.
5. Keep `REQUIRE_GPU=True`, `FEATURE_WORKERS=2`, `GPU_BATCH_SIZE=256`, `MODE='full'`.
   Run all. The configuration must show `/GPU:0`; otherwise stop and enable the accelerator.

The first stage remains CPU preprocessing, now optimized and parallel. It prints
progress and estimated remaining time. Low GPU use before `V6 LSTM device: /GPU:0`
is expected, not evidence of a CPU fallback. No speedup is guaranteed on Kaggle until
measured there. V6 creates new run/cache identities rather than trusting old shards.

## Outputs and holdouts
Outputs: `/kaggle/working/spendly_v6/`. Download the newest `run_<id>_results.zip` and
executed notebook and send them for analysis. Do not attach/score holdouts until the
development results have been reviewed. All existing frozen/final guards remain.
After final evaluation download the existing ZIP and ledger immediately; do NOT use
Save & Run All to preserve a final run (it executes the tests again).


In [ ]:
def find_kaggle_file(expected_name, root='/kaggle/input'):
    """Resolve one attached input only when requested; do not open test files early."""
    root = Path(root)
    if not root.is_dir():
        raise FileNotFoundError('Kaggle input directory missing. Attach the private R3 dataset with Add Input / Add Data.')
    names = (expected_name, expected_name.replace('.csv', '.zip'))
    for name in dict.fromkeys(names):
        matches = sorted(p for p in root.rglob(name) if p.is_file())
        if len(matches) > 1:
            raise ValueError('Multiple copies of '+name+' are attached. Set the explicit path in COHORT_PATHS/MANIFEST_PATH.')
        if matches:
            return matches[0]
    raise FileNotFoundError('Missing '+expected_name+' (or its ZIP) under /kaggle/input. Use Add Input / Upload to attach it.')
"""Embedded GPU preflight and explicit cuDNN-compatible LSTM construction."""


def configure_training_device(tf, require_gpu=True):
    devices = tf.config.list_physical_devices('GPU')
    if require_gpu and not devices:
        raise RuntimeError(
            'No TensorFlow GPU is available. In Colab select Runtime > Change runtime type > '
            'Hardware accelerator > T4 GPU (or another GPU) > Save, reconnect, and run '
            'the configuration cell again. A notebook cannot allocate a GPU from a CPU '
            'runtime. If Colab reports a GPU quota limit, wait for availability. '
            'Set REQUIRE_GPU=False only for intentional CPU debugging.')
    for device in devices:
        try:
            tf.config.experimental.set_memory_growth(device, True)
        except RuntimeError:
            pass  # May be a rerun after device initialization; probe that device below.
    target = '/GPU:0' if devices else '/CPU:0'
    previous = tf.config.get_soft_device_placement()
    try:
        tf.config.set_soft_device_placement(False)
        with tf.device(target):
            probe = tf.linalg.matmul(tf.ones((32, 32)), tf.ones((32, 32)))
        probe.numpy()
    finally:
        tf.config.set_soft_device_placement(previous)
    if devices and 'GPU:' not in probe.device.upper():
        raise RuntimeError('GPU was listed but the test operation did not execute on a GPU')
    details = []
    for device in devices:
        info = tf.config.experimental.get_device_details(device)
        details.append(dict(device=device.name, name=info.get('device_name', 'GPU'),
                            compute_capability=list(info.get('compute_capability', ()))))
    return dict(training_device=target, gpu_required=bool(require_gpu), gpus=details,
                probe_device=probe.device, tensorflow_version=tf.__version__,
                scope=('GPU operation verified' if devices else 'CPU debug operation verified') +
                      '; pandas/features and sklearn IF remain CPU')


def make_accelerated_lstm(tf, spec, context_width, require_gpu=True):
    """Float32 retained; require fused cuDNN on GPU where the Keras API supports it."""
    import inspect
    if tf.keras.mixed_precision.global_policy().name != 'float32':
        raise ValueError('Use float32 for this run; mixed precision needs a separately identified experiment')
    runtime = configure_training_device(tf, require_gpu)
    options = dict(units=spec['units'], activation='tanh', recurrent_activation='sigmoid',
                   use_bias=True, dropout=0., recurrent_dropout=0., unroll=False,
                   kernel_regularizer=tf.keras.regularizers.l2(spec['regularization']))
    explicit_cudnn = 'use_cudnn' in inspect.signature(tf.keras.layers.LSTM.__init__).parameters
    if explicit_cudnn:
        options['use_cudnn'] = bool(runtime['gpus'])
    with tf.device(runtime['training_device']):
        seq = tf.keras.Input((8, 2), name='weekly_history', dtype='float32')
        context = tf.keras.Input((context_width,), name='recurrence_context', dtype='float32')
        encoded = tf.keras.layers.LSTM(**options)(seq)
        hidden = tf.keras.layers.Dense(16, activation='relu')(tf.keras.layers.Concatenate()([encoded, context]))
        residual = tf.keras.layers.Dense(1, kernel_initializer='zeros', bias_initializer='zeros', dtype='float32')(hidden)
        model = tf.keras.Model([seq, context], residual)
        model.compile(optimizer=tf.keras.optimizers.Adam(spec['learning_rate'], clipnorm=1.), loss='mae', jit_compile=False)
        test_output = model([tf.zeros((2, 8, 2)), tf.zeros((2, context_width))], training=False)
        test_output.numpy()
    if runtime['gpus'] and 'GPU:' not in test_output.device.upper():
        raise RuntimeError('LSTM forward output was not on GPU; refusing silent CPU training')
    runtime.update(lstm_output_device=test_output.device,
                   cudnn_mode='required' if runtime['gpus'] and explicit_cudnn else 'automatic' if runtime['gpus'] else 'cpu_debug',
                   precision='float32')
    return model, runtime


In [ ]:
import os, sys, io, json, hashlib, time, importlib.util, importlib.metadata
from pathlib import Path
from collections import defaultdict, deque

MODE = 'full'  # 'smoke' is software verification only; still reads every uploaded user
REQUIRE_GPU = True  # V6 refuses unintended CPU LSTM training
GPU_BATCH_SIZE = 256  # Try 512 only as a NEW development run; changes training semantics/run identity
GOAL = .80  # reporting reference only, not a guaranteed result or release criterion
WORK_DIR = Path('/kaggle/working/spendly_v6') if Path('/kaggle/working').exists() else Path('spendly_v6_kaggle_runs')
# Optional paths for files ALREADY in Colab/Drive; empty means discover attached Kaggle input.
MANIFEST_PATH = ''
COHORT_PATHS = {'train':'', 'validation':'', 'calibration':'', 'test':'', 'test_shifted':''}
DOWNLOAD_RESULTS = False
if MODE not in ('full','smoke'):
    raise ValueError('MODE must be full or smoke')
required={'numpy':'numpy','pandas':'pandas','sklearn':'scikit-learn','matplotlib':'matplotlib','joblib':'joblib','tensorflow':'tensorflow'}
missing=[dist for module,dist in required.items() if importlib.util.find_spec(module) is None]
if missing:
    raise RuntimeError('Missing packages: '+str(missing)+'. Use a compatible Kaggle notebook image; no automatic SDK/package upgrades.')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_recall_curve, average_precision_score, roc_auc_score, r2_score
from threadpoolctl import threadpool_limits
THREAD_LIMIT = threadpool_limits(limits=2)
VERSIONS={dist:importlib.metadata.version(dist) for dist in required.values()}
FEATURE_CODE_HASH = '54fa0c12966b5789723774f16772d86745e9ad8210aad7485b2d768c2fe1e744'
COLUMN_MAP = {}
BLANK_LABELS_ARE_NORMAL = False  # R3 has explicit labels in every row
RUNTIME=configure_training_device(tf,REQUIRE_GPU)
FEATURE_WORKERS = 2
print('LSTM runtime:',json.dumps(RUNTIME,indent=2))
print('Stage map: CSV/features = CPU; LSTM epochs =',RUNTIME['training_device'],'; Isolation Forest = CPU.')
print('Versions:', VERSIONS)
WORK_DIR.mkdir(parents=True,exist_ok=True)
print('Workspace:', WORK_DIR)

print('V6 uses GPU0 for LSTM; GPU1 is intentionally unused. Feature/IF work stays on CPU.')
PREVIOUS_STATE_DIRS = ['/kaggle/working/spendly_r3']  # add any restored V5/Colab work directories
HOLDOUT_STATE_DIR = Path('/kaggle/working/spendly_holdout_state') if Path('/kaggle/working').exists() else WORK_DIR.parent/'spendly_holdout_state'


In [ ]:
def parse_dates(values):
    return pd.to_datetime(values, errors='raise', utc=True, format='mixed').dt.tz_convert('Africa/Nairobi').dt.tz_localize(None)

def clean_data(raw, blank_normal=False):
    data = raw.rename(columns=COLUMN_MAP).copy()
    if not data.columns.is_unique:
        raise ValueError('COLUMN_MAP creates duplicate columns')
    required_columns = {'user_id', 'transaction_timestamp', 'amount', 'category'}
    missing_columns = required_columns - set(data.columns)
    if missing_columns:
        raise ValueError('Missing columns: ' + ', '.join(sorted(missing_columns)))
    for column in ('user_id', 'category'):
        if data[column].isna().any() or data[column].astype(str).str.strip().eq('').any():
            raise ValueError(column + ' contains missing values')
        data[column] = data[column].astype(str).str.strip()
    data['transaction_timestamp'] = parse_dates(data.transaction_timestamp)
    data['amount'] = pd.to_numeric(data.amount, errors='raise')
    if data.transaction_timestamp.isna().any() or not np.isfinite(data.amount).all() or data.amount.lt(0).any():
        raise ValueError('Dates must be present and amounts finite and non-negative KES')
    if 'transaction_id' not in data:
        data['transaction_id'] = ['row_' + str(i) for i in range(len(data))]
    if data.transaction_id.isna().any():
        raise ValueError('transaction_id contains missing values')
    data['transaction_id'] = data.transaction_id.astype(str).str.strip()
    if not data.transaction_id.is_unique or data.transaction_id.eq('').any():
        raise ValueError('transaction_id must be nonblank and unique; resolve duplicate records first')
    if 'transaction_type' not in data:
        data['transaction_type'] = 'expense'
    data['transaction_type'] = data.transaction_type.astype(str).str.strip().str.lower()
    if not data.transaction_type.isin(['expense', 'income']).all():
        raise ValueError('transaction_type must be expense or income')
    labels = data.get('is_anomaly', pd.Series(pd.NA, index=data.index)).astype('string').str.strip().str.lower()
    mapping = {'true': 1., '1': 1., '1.0': 1., 'yes': 1., 'false': 0., '0': 0., '0.0': 0., 'no': 0.}
    if (labels.notna() & labels.ne('') & ~labels.isin(mapping)).any():
        raise ValueError('is_anomaly contains invalid labels; use 0/1, true/false, yes/no or blank')
    data['label'] = labels.map(mapping).astype(float)
    if blank_normal:
        if 'is_anomaly' not in data:
            raise ValueError('Cannot infer normal labels when is_anomaly is entirely absent')
        data['label'] = data.label.fillna(0.)
    data['merchant'] = data.get('merchant', pd.Series('', index=data.index)).fillna('').astype(str).str.strip()
    data['event_id'] = data.get('event_id', pd.Series(pd.NA, index=data.index)).astype('string').str.strip().replace('', pd.NA)
    coverage = {'observation_start', 'observation_end'} & set(data.columns)
    if coverage and len(coverage) != 2:
        raise ValueError('Provide both observation_start and exclusive observation_end')
    for column in coverage:
        data[column] = parse_dates(data[column])
        if data[column].isna().any() or data.groupby('user_id')[column].nunique().ne(1).any():
            raise ValueError(column + ' must be present and constant per user')
    if coverage and ((data.transaction_timestamp < data.observation_start) | (data.transaction_timestamp >= data.observation_end)).any():
        raise ValueError('Transactions fall outside declared observation coverage')
    if 'anomaly_type' in data:
        data['family'] = data['anomaly_type'].fillna('unspecified').astype(str)
    expense = data.loc[data.transaction_type.eq('expense')].copy()
    if expense.empty:
        raise ValueError('No expense records found')
    return expense.sort_values(['user_id', 'transaction_timestamp', 'transaction_id']).reset_index(drop=True)


In [ ]:
A_FEATURES = ['amount_ratio', 'category_ratio', 'category_deviation', 'merchant_ratio', 'count_hour', 'sum_hour_ratio', 'count_day', 'sum_week_ratio', 'duplicate_hour', 'merchants_hour', 'night_rarity', 'category_rarity', 'user_cold', 'merchant_missing']

def anomaly_features(data):
    values = []
    for _, group in data.groupby('user_id', sort=False):
        history = deque(maxlen=100)
        categories, merchants = defaultdict(lambda: deque(maxlen=40)), defaultdict(lambda: deque(maxlen=20))
        recent, category_counts, seen, nights = deque(), defaultdict(int), 0, 0
        for t, batch in group.groupby('transaction_timestamp', sort=True):
            while recent and recent[0][0] < t - pd.Timedelta(days=7):
                recent.popleft()
            hour = [r for r in recent if r[0] >= t - pd.Timedelta(hours=1)]
            day = [r for r in recent if r[0] >= t - pd.Timedelta(days=1)]
            for row in batch.itertuples():
                a, c, m = row.amount, row.category, row.merchant
                typical = max(float(np.median(history)) if history else 1., 1.)
                cat = categories[c]
                center = max(float(np.median(cat)) if cat else typical, 1.)
                spread = max(float(np.median(np.abs(np.asarray(cat) - center))) * 1.4826 if cat else 0., center * .20, 1.)
                merchant_center = max(float(np.median(merchants[m])) if m and merchants[m] else center, 1.)
                night = t.hour < 5 or t.hour >= 23
                features = [a / typical, a / center, max(0., (a - center) / spread), a / merchant_center, len(hour),
                    sum(r[1] for r in hour) / typical, len(day), sum(r[1] for r in recent) / typical,
                    sum(bool(m) and r[2] == m and r[3] == c and abs(r[1] - a) < .01 for r in hour),
                    len({r[2] for r in hour if r[2]}), float(night) * (1 - (nights + 1) / (seen + 2)),
                    1 - category_counts[c] / max(1, seen), float(seen < 10), float(not m)]
                values.append((row.Index, features))
            for row in batch.itertuples():
                history.append(row.amount); categories[row.category].append(row.amount)
                if row.merchant:
                    merchants[row.merchant].append(row.amount)
                recent.append((t, row.amount, row.merchant, row.category))
                seen += 1; nights += int(t.hour < 5 or t.hour >= 23); category_counts[row.category] += 1
    return pd.DataFrame([v for _, v in values], index=[i for i, _ in values], columns=A_FEATURES).reindex(data.index)

def time_boundaries(data):
    start = data.transaction_timestamp.min().normalize() + pd.Timedelta(weeks=8)
    end = data.transaction_timestamp.max().normalize()
    if end - start < pd.Timedelta(weeks=12):
        raise ValueError('Provide at least 20 weeks of history for reliable four-way temporal splitting')
    return [start + (end - start) * fraction for fraction in (.55, .70, .80)]

def split_masks(frame, cuts, forecast=False):
    t = frame.time if forecast else frame.transaction_timestamp
    end = frame.end if forecast else t
    masks = [((t < cuts[0]) & (end <= cuts[0])), ((t >= cuts[0]) & (t < cuts[1]) & (end <= cuts[1])),
             ((t >= cuts[1]) & (t < cuts[2]) & (end <= cuts[2])), t >= cuts[2]]
    if not forecast:
        # Include event IDs independently of labels. Events outside a mask cannot leak into selection.
        key = pd.Series(list(zip(frame.user_id, frame.event_id.fillna(''))), index=frame.index)
        totals = key.loc[frame.event_id.notna()].value_counts()
        for i in range(3):
            counts = key.loc[masks[i] & frame.event_id.notna()].value_counts()
            contained = counts.index[counts.eq(totals.reindex(counts.index))]
            masks[i] &= frame.event_id.isna() | key.isin(contained)
    if any(not mask.any() for mask in masks):
        raise ValueError('An empty temporal split remains; supply longer/more complete history')
    return dict(zip(('train', 'validation', 'calibration', 'test'), masks))


In [ ]:
def classification_metrics(labels, scores, threshold):
    labels, scores = np.asarray(labels, float), np.asarray(scores, float)
    known = np.isfinite(labels)
    y, alerts = labels[known].astype(bool), scores[known] >= threshold
    tp, fp, tn, fn = int((y & alerts).sum()), int((~y & alerts).sum()), int((~y & ~alerts).sum()), int((y & ~alerts).sum())
    both = y.any() and (~y).any()
    p = tp / (tp + fp) if tp + fp else (0. if len(y) else np.nan)
    r = tp / (tp + fn) if tp + fn else np.nan
    f1 = 2 * tp / (2 * tp + fp + fn) if y.any() else np.nan
    result = dict(rows=len(labels), labelled_rows=len(y), unknown_labels=int((~known).sum()), positive_support=int(y.sum()),
        accuracy=(tp + tn) / len(y) if len(y) else np.nan, balanced_accuracy=(r + tn / (tn + fp)) / 2 if both else np.nan,
        precision=p, recall=r, F1=f1, AP=float(average_precision_score(y, scores[known])) if both else np.nan,
        ROC_AUC=float(roc_auc_score(y, scores[known])) if both else np.nan, FPR=fp / (fp + tn) if fp + tn else np.nan,
        TP=tp, FP=fp, TN=tn, FN=fn, alert_rate=float(np.mean(scores >= threshold)) if len(scores) else np.nan)
    result['negative_support'] = int((~y).sum())
    result['metric_status'] = 'both labelled classes; metrics conditional on label coverage' if both else 'incomplete label classes: precision/F1/accuracy unavailable'
    if not both:
        for key in ('precision', 'F1', 'accuracy'):
            result[key] = np.nan
    result['all_80_targets_met'] = bool(both and all(result[k] >= GOAL for k in ('accuracy', 'balanced_accuracy', 'precision', 'recall', 'F1')))
    return result

def select_threshold(labels, scores, fallback):
    labels, scores = np.asarray(labels, float), np.asarray(scores, float)
    known = np.isfinite(labels)
    if len(np.unique(labels[known])) < 2:
        return dict(threshold=float(fallback), status='uncalibrated: requires both labelled classes')
    p, r, thresholds = precision_recall_curve(labels[known], scores[known])
    p, r = p[:-1], r[:-1]
    f1 = 2 * p * r / np.maximum(p + r, 1e-15)
    eligible = np.flatnonzero((p >= GOAL) & (r >= GOAL))
    status = 'precision/recall goal met on this slice' if len(eligible) else '80% goal unmet; best F1 fallback'
    pool = eligible if len(eligible) else np.arange(len(thresholds))
    best = max(pool, key=lambda i: (f1[i], r[i], p[i]))
    return dict(threshold=float(thresholds[best]), status=status)

def regression_metrics(y, prediction):
    y, prediction = np.asarray(y, float), np.asarray(prediction, float)
    error = np.abs(y - prediction)
    wape = float(error.sum() / y.sum()) if y.sum() else np.nan
    # For actual zero, within-20% requires a zero prediction (no hidden monetary tolerance).
    within = float(np.mean(error <= .20 * np.abs(y) + 1e-9))
    return dict(rows=len(y), MAE_KES=float(error.mean()), RMSE_KES=float(np.sqrt(np.mean((y - prediction) ** 2))),
                actual_mean_KES=float(y.mean()), predicted_mean_KES=float(prediction.mean()), bias_KES=float((prediction - y).mean()),
                WAPE=wape, R2=float(r2_score(y, prediction)) if len(y) > 1 and np.var(y) > 0 else np.nan,
                within_20_percent=within, WAPE_goal_met=bool(wape <= .20), within_20_goal_met=bool(within >= GOAL))


In [ ]:
def json_safe(value):
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, (np.integer, np.bool_)):
        return value.item()
    if isinstance(value, (float, np.floating)):
        return float(value) if np.isfinite(value) else None
    return value


In [ ]:
def infer_schedule(group, minimum=2):
    """Infer a monthly or weekly schedule using observed history only.

    Category pooling can recover a bill whose merchant changes. Multiple purchases
    in one month are not interpreted as a monthly bill. No category names/labels.
    """
    dates = pd.DatetimeIndex(group.transaction_timestamp).unique().sort_values()
    if len(dates) < minimum:
        return None
    months = dates.year * 12 + dates.month
    monthly = months.is_unique and dates.day.max() - dates.day.min() <= 3
    gaps = np.diff(dates.to_numpy()) / np.timedelta64(1, 'D')
    if monthly and np.median(np.diff(months)) <= 2 and np.max(np.diff(months)) <= 3:
        step, period = 'month', 30.
        # Scheduled day comes from history, not the generator's rent/payday constants.
        anchor = int(round(float(np.median(dates.day))))
    else:
        period = float(np.median(gaps))
        if not 5 <= period <= 16 or np.median(np.abs(gaps - period)) > .12 * period:
            return None
        step, anchor = 'days', None
    amounts = group.sort_values('transaction_timestamp').amount.tail(6)
    amount = float(amounts.median())
    if amount <= 0 or float(np.median(np.abs(amounts - amount))) > .30 * amount:
        return None
    return dict(last=dates[-1], amount=amount, step=step, period=period, anchor=anchor,
                confidence=min(1., (len(dates) - 1) / 2))

def next_occurrence(schedule, current):
    if schedule['step'] == 'month':
        month = current.normalize().replace(day=1) + pd.DateOffset(months=1)
        day = min(schedule['anchor'], month.days_in_month)
        return month.replace(day=day) + (schedule['last'] - schedule['last'].normalize())
    return current + pd.Timedelta(days=round(schedule['period']))

def schedule_features(history, left, right, lag_start, pooling=True, minimum=2):
    due, recurring_ids, confidence = 0., set(), []
    # Prefer category pooling only if the entire category is demonstrably periodic.
    groups = []
    for _, category in history.groupby('category', sort=True):
        schedule = infer_schedule(category, minimum) if pooling else None
        if schedule is not None:
            groups.append((category, schedule))
        else:
            groups.extend((merchant, infer_schedule(merchant, minimum))
                          for _, merchant in category.groupby('merchant', sort=True))
    for group, schedule in groups:
        if schedule is None:
            continue
        next_due = next_occurrence(schedule, schedule['last'])
        if next_due < left - pd.Timedelta(days=40):
            continue
        while next_due < right:
            if next_due >= left:
                due += schedule['amount']
                confidence.append(schedule['confidence'])
            next_due = next_occurrence(schedule, next_due)
        recurring_ids.update(group.loc[group.transaction_timestamp.ge(lag_start), 'transaction_id'])
    return due, recurring_ids, float(np.mean(confidence)) if confidence else 0.

V5_F_FEATURES = ([f'lag{i}' for i in range(1, 9)] +
                 [f'nonrec_lag{i}' for i in range(1, 9)] +
                 ['mean4', 'std8', 'recurring', 'nonrecurring', 'log_scale',
                  'nonrec_median', 'nonrec_std', 'due_confidence', 'count4', 'zero_fraction'])

def forecast_features_v5(data, variant='pooled'):
    rows, values, skipped = [], [], 0
    for user, group in data.groupby('user_id', sort=True):
        start = group.observation_start.iloc[0] if 'observation_start' in group else group.transaction_timestamp.min().normalize()
        end = group.observation_end.iloc[0] if 'observation_end' in group else group.transaction_timestamp.max().normalize()
        first = start.normalize() - pd.Timedelta(days=start.weekday())
        if first < start:
            first += pd.Timedelta(weeks=1)
        last = end.normalize() - pd.Timedelta(days=end.weekday())
        boundaries = pd.date_range(first, last, freq='7D')
        if len(boundaries) < 10:
            skipped += 1
            continue
        week = group.transaction_timestamp.dt.normalize() - pd.to_timedelta(group.transaction_timestamp.dt.weekday, unit='D')
        totals = group.groupby(week).amount.agg(['sum', 'count']).reindex(boundaries[:-1], fill_value=0.)
        weekly = totals['sum'].to_numpy()
        for i in range(8, len(weekly)):
            left, right = boundaries[i], boundaries[i + 1]
            lag_start = boundaries[i - 8]
            past = weekly[i - 8:i]
            scale = max(float(np.mean(past)), 1.)  # stable even with sparse/bill-heavy weeks
            history = group.loc[group.transaction_timestamp.lt(left) & group.transaction_timestamp.ge(left - pd.Timedelta(weeks=26))]
            due, ids, confidence = schedule_features(history, left, right, lag_start, pooling=variant == 'pooled')
            discretionary = history.loc[history.transaction_timestamp.ge(lag_start) & ~history.transaction_id.isin(ids)]
            dweek = discretionary.transaction_timestamp.dt.normalize() - pd.to_timedelta(discretionary.transaction_timestamp.dt.weekday, unit='D')
            nonrec = discretionary.groupby(dweek).amount.sum().reindex(boundaries[i - 8:i], fill_value=0.).to_numpy()
            baseline_mean, baseline_median = due + float(nonrec.mean()), due + float(np.median(nonrec))
            values.append([*(past[::-1] / scale), *(nonrec[::-1] / scale), past[-4:].mean() / scale,
                           past.std() / scale, due / scale, nonrec.mean() / scale, np.log(scale),
                           np.median(nonrec) / scale, nonrec.std() / scale, confidence,
                           float(totals['count'].iloc[i - 4:i].mean()), np.mean(past == 0)])
            rows.append(dict(user_id=user, time=left, end=right, y=weekly[i], scale=scale, last=past[-1],
                             mean4=past[-4:].mean(), recurring=baseline_mean, robust_recurring=baseline_median,
                             due=due, nonrec_mean=nonrec.mean()))
    return pd.DataFrame(rows), pd.DataFrame(values, columns=V5_F_FEATURES), skipped

def enhanced_anomaly_features(data):
    x = anomaly_features(data)
    # Keep the original causal inputs, but supply focused feature views to Isolation Forest.
    x['category_excess'] = np.maximum(0., x.category_ratio - 1.)
    x['merchant_excess'] = np.maximum(0., x.merchant_ratio - 1.)
    x['rare_large'] = x.category_rarity * np.log1p(x.category_excess) * (1 - x.user_cold)
    x['burst_amount'] = x.count_hour * np.log1p(x.amount_ratio)
    return x

def _lstm_inputs_v5(x, scales):
    seq = np.stack([x[[f'lag{i}' for i in range(8, 0, -1)]].to_numpy(),
                    x[[f'nonrec_lag{i}' for i in range(8, 0, -1)]].to_numpy()], axis=-1)
    seq = scales['sequence'].transform(seq.reshape(-1, 2)).reshape(-1, 8, 2)
    context = scales['context'].transform(x.iloc[:, 16:])
    return seq.astype('float32'), context.astype('float32')

def _lstm_scalers_v5(x):
    seq = np.stack([x[[f'lag{i}' for i in range(8, 0, -1)]].to_numpy(),
                    x[[f'nonrec_lag{i}' for i in range(8, 0, -1)]].to_numpy()], axis=-1)
    return dict(sequence=StandardScaler().fit(seq.reshape(-1, 2)), context=StandardScaler().fit(x.iloc[:, 16:]))

def predict_forecast_v5(spec, fitted, x, frame):
    if spec['kind'] != 'lstm':
        raise ValueError('Only LSTM is permitted for forecasting')
    inputs = _lstm_inputs_v5(x, fitted['scales'])
    prediction = np.concatenate([fitted['model']([inputs[0][i:i + 256], inputs[1][i:i + 256]], training=False).numpy().ravel()
                                 for i in range(0, len(frame), 256)])
    return np.maximum(0., (prediction + fitted.get('bias_correction', 0.)) * frame.scale.to_numpy() + frame.recurring.to_numpy())


In [ ]:
"""Causal user-relative burst and recurring-change features for Isolation Forest.

Declared in RHYTHM_PROTOCOL.md. Strictly past-only, label-blind and nonnegative so
the existing log1p scoring contract is preserved. No new model family is introduced.
"""

RHYTHM_FEATURES = ['day_count_excess', 'week_count_excess', 'gap_compression',
                   'recurring_increase', 'recurring_known']


def _spread(values):
    """Robust MAD-based spread with a one-transaction floor for sparse users."""
    if not len(values):
        return 1.
    array = np.asarray(values, dtype=float)
    centre = float(np.median(array))
    return max(float(np.median(np.abs(array - centre))) * 1.4826, 1.)


def rhythm_features(data):
    """One forward pass per user; simultaneous transactions share only earlier history."""
    rows = []
    for _, group in data.groupby('user_id', sort=False):
        day_counts, week_counts, gaps = deque(maxlen=90), deque(maxlen=26), deque(maxlen=200)
        amounts, times = defaultdict(lambda: deque(maxlen=12)), defaultdict(lambda: deque(maxlen=12))
        recent = deque()
        current_day = current_week = None
        day_total = week_total = 0
        previous = None
        for t, batch in group.groupby('transaction_timestamp', sort=True):
            day, week = t.normalize(), t.normalize() - pd.Timedelta(days=t.weekday())
            if current_day is None:
                current_day, current_week = day, week
            # Completed periods only; the in-progress day/week never enters its own baseline.
            while current_day < day:
                day_counts.append(day_total)
                current_day, day_total = current_day + pd.Timedelta(days=1), 0
            while current_week < week:
                week_counts.append(week_total)
                current_week, week_total = current_week + pd.Timedelta(weeks=1), 0
            while recent and recent[0] < t - pd.Timedelta(days=7):
                recent.popleft()
            day_window = sum(1 for r in recent if r >= t - pd.Timedelta(days=1))
            week_window = len(recent)
            day_median = float(np.median(day_counts)) if day_counts else 0.
            week_median = float(np.median(week_counts)) if week_counts else 0.
            day_excess = max(0., (day_window - day_median) / _spread(day_counts))
            week_excess = max(0., (week_window - week_median) / _spread(week_counts))
            gap = (t - previous).total_seconds() / 60 if previous is not None else None
            compression = 0.
            if gap is not None and gaps:
                compression = min(float(np.median(gaps)) / max(gap, 1.), 100.)
            for row in batch.itertuples():
                key = (row.category, row.merchant)
                past_times, past_amounts = times[key], amounts[key]
                increase, known = 0., 0.
                if len(past_times) >= 3:
                    intervals = np.diff(pd.DatetimeIndex(past_times).to_numpy()) / np.timedelta64(1, 'D')
                    cadence = float(np.median(intervals))
                    if 25 <= cadence <= 35 and float(np.median(np.abs(intervals - cadence))) <= .2 * cadence:
                        expected = float(np.median(past_amounts))
                        if expected > 0:
                            known, increase = 1., max(0., row.amount / expected - 1.)
                rows.append((row.Index, [day_excess, week_excess, compression, increase, known]))
            for row in batch.itertuples():
                times[(row.category, row.merchant)].append(t)
                amounts[(row.category, row.merchant)].append(row.amount)
                recent.append(t)
                day_total += 1
                week_total += 1
            # Record the gap only after scoring, so it cannot inform its own baseline.
            if gap is not None:
                gaps.append(gap)
            previous = t
    return pd.DataFrame([values for _, values in rows], index=[index for index, _ in rows],
                        columns=RHYTHM_FEATURES, dtype=float).reindex(data.index)


def rhythm_anomaly_features(data):
    """Existing enhanced causal features joined with the targeted rhythm candidates."""
    return enhanced_anomaly_features(data).join(rhythm_features(data))


def rhythm_views(x):
    """Existing views plus the two declared candidate views."""
    views = dict(all=[c for c in x.columns if c not in RHYTHM_FEATURES],
                 context=['category_ratio', 'category_deviation', 'merchant_ratio',
                          'count_hour', 'duplicate_hour', 'rare_large', 'burst_amount'],
                 excess=['category_excess', 'merchant_excess', 'duplicate_hour', 'rare_large', 'burst_amount'])
    views['rhythm'] = views['excess'] + RHYTHM_FEATURES
    views['rhythm_only'] = list(RHYTHM_FEATURES)
    return views


In [ ]:
"""Alert-budget-aware threshold selection for sparse calibration slices.

Declared in CALIBRATION_PROTOCOL.md. The alert budget is a product constraint, so it
is enforced as a floor on the threshold rather than left to emerge from best-F1
optimisation on a handful of calibration positives.
"""


def alert_rate_floor(scores, max_alert_rate):
    """Most permissive threshold whose realised alert rate stays within the budget.

    Quantiles are not sufficient: with ties or small slices, `scores >= quantile`
    can alert more rows than the budget allows, so the count is bounded directly.
    """
    scores = np.asarray(scores, dtype=float)
    if not 0 < max_alert_rate < 1:
        raise ValueError('Alert budget must be a fraction strictly between 0 and 1')
    if not len(scores):
        raise ValueError('Cannot derive an alert budget from an empty calibration slice')
    ordered = np.sort(scores)
    budget = int(np.floor(max_alert_rate * len(ordered)))
    if budget == 0:
        # The budget cannot afford a single alert on this slice.
        return float(np.nextafter(ordered[-1], np.inf))
    unique = np.unique(ordered)
    alerts = len(ordered) - np.searchsorted(ordered, unique, side='left')
    affordable = unique[alerts <= budget]
    if not len(affordable):
        return float(np.nextafter(ordered[-1], np.inf))
    return float(affordable[0])


def false_positive_floor(labels, scores, max_false_positive_rate):
    """Most permissive threshold whose calibration false-positive rate stays in budget.

    The release gate constrains false positives among negatives, not total alerts.
    At 3% prevalence those differ by roughly an order of magnitude, so constraining
    alert rate instead needlessly caps recall.
    """
    labels = np.asarray(labels, dtype=float)
    scores = np.asarray(scores, dtype=float)
    if not 0 < max_false_positive_rate < 1:
        raise ValueError('False-positive budget must be a fraction strictly between 0 and 1')
    negatives = scores[labels == 0]
    if not len(negatives):
        raise ValueError('A false-positive budget requires labelled negatives')
    ordered = np.sort(negatives)
    budget = int(np.floor(max_false_positive_rate * len(ordered)))
    if budget == 0:
        return float(np.nextafter(ordered[-1], np.inf))
    unique = np.unique(ordered)
    false_alerts = len(ordered) - np.searchsorted(ordered, unique, side='left')
    affordable = unique[false_alerts <= budget]
    if not len(affordable):
        return float(np.nextafter(ordered[-1], np.inf))
    return float(affordable[0])


def fpr_capped_threshold(labels, scores, fallback, max_false_positive_rate, minimum_positives=None):
    """Maximise recall subject to the false-positive budget the gate actually imposes."""
    labels = np.asarray(labels, dtype=float)
    scores = np.asarray(scores, dtype=float)
    positives = int(np.nansum(labels == 1))
    negatives = int(np.nansum(labels == 0))
    if not negatives or (minimum_positives is not None and positives < minimum_positives):
        # Without usable labels the budget cannot be measured; fall back to the
        # large, label-free training distribution rather than guessing.
        chosen, status = float(fallback), f'support fallback: {positives} positives, {negatives} negatives'
    else:
        chosen = false_positive_floor(labels, scores, max_false_positive_rate)
        status = f'within {max_false_positive_rate:.3%} calibration false-positive budget'
    known = np.isfinite(labels)
    realised = float(np.mean(scores[known & (labels == 0)] >= chosen)) if negatives else float('nan')
    return dict(threshold=chosen, status=status, calibration_false_positive_rate=realised,
                calibration_alert_rate=float(np.mean(scores >= chosen)) if len(scores) else float('nan'),
                calibration_positives=positives)


def capped_threshold(labels, scores, fallback, max_alert_rate=None, minimum_positives=None):
    """Iteration 1 rule: total-alert-rate budget. Retained for comparison only.

    Superseded by fpr_capped_threshold; see CALIBRATION_PROTOCOL.md iteration 2 for
    why constraining total alert rate caps recall at low prevalence.
    """
    labels = np.asarray(labels, dtype=float)
    scores = np.asarray(scores, dtype=float)
    positives = int(np.nansum(labels == 1))
    reasons = []
    if minimum_positives is not None and positives < minimum_positives:
        # Too few positives to trust a label-driven threshold; use the fitted
        # training distribution instead of optimising against noise.
        chosen = float(fallback)
        reasons.append(f'support fallback: {positives} calibration positives')
    else:
        selected = select_threshold(labels, scores, fallback)
        chosen = float(selected['threshold'])
        reasons.append(selected['status'])
    if max_alert_rate is not None:
        floor = alert_rate_floor(scores, max_alert_rate)
        if floor > chosen:
            chosen = floor
            reasons.append(f'raised to {max_alert_rate:.3%} alert budget')
        else:
            reasons.append(f'within {max_alert_rate:.3%} alert budget')
    realised = float(np.mean(scores >= chosen)) if len(scores) else float('nan')
    return dict(threshold=chosen, status='; '.join(reasons),
                calibration_alert_rate=realised, calibration_positives=positives)


In [ ]:
"""Standalone Colab orchestration embedded by build_colab_r3.py.

Features, metrics, rules and numerical imports are embedded earlier in the notebook.
This module deliberately performs no work on import.
"""

def file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def json_write(path, value):
    path=Path(path)
    content=json.dumps(json_safe(value), indent=2, sort_keys=True, allow_nan=False)
    temporary=path.with_name(path.name+'.writing')
    temporary.write_text(content,encoding='utf-8')
    temporary.replace(path)

def read_r3_manifest(path):
    manifest = json.loads(Path(path).read_text(encoding='utf-8'))
    names = {'train', 'validation', 'calibration', 'test', 'test_shifted'}
    if manifest.get('version') != 'spendly-synthetic-r3-v1' or manifest.get('status') != 'complete':
        raise ValueError('Upload the complete R3 dataset manifest.json, not an old model/result manifest')
    if set(manifest.get('cohorts', {})) != names or manifest.get('currency') != 'KES' or manifest.get('timezone') != 'Africa/Nairobi':
        raise ValueError('Unexpected R3 cohort/currency/timezone contract')
    if manifest.get('weeks_per_user', 0) < 24:
        raise ValueError('At least 24 weeks of history required')
    for name, entry in manifest['cohorts'].items():
        if not isinstance(entry.get('sha256'), str) or len(entry['sha256']) != 64 or entry.get('users', 0) < 1:
            raise ValueError('Invalid cohort hash or user count: ' + name)
    if len({e['seed'] for e in manifest['cohorts'].values()}) != 5:
        raise ValueError('Dataset cohorts must have distinct seeds')
    return manifest

def request_one_file(label, expected_name, local_path='', **kwargs):
    print(label)
    path=Path(local_path) if local_path else find_kaggle_file(expected_name)
    if not path.is_file():
        raise FileNotFoundError('Attached input missing: '+str(path))
    print('Using attached Kaggle input:',path)
    return path

def verify_cohort_file(path, cohort, manifest, directory):
    """Stream exactly cohort.csv from a ZIP; do not extract arbitrary archive paths."""
    import zipfile
    path, directory = Path(path), Path(directory)
    expected = manifest['cohorts'][cohort]
    directory.mkdir(parents=True, exist_ok=True)
    output = directory / (cohort + '.csv')
    if path.suffix.lower() == '.zip':
        with zipfile.ZipFile(path) as zipped:
            members = [info for info in zipped.infolist() if not info.is_dir()]
            if len(members) != 1 or members[0].filename != cohort + '.csv':
                raise ValueError('ZIP must contain only ' + cohort + '.csv at its root')
            info = members[0]
            if info.file_size > 1_500_000_000 or info.file_size != expected.get('csv_bytes', info.file_size):
                raise ValueError('Unexpected uncompressed CSV size')
            temporary = directory / (cohort + '.partial')
            try:
                with zipped.open(info) as source, temporary.open('wb') as dest:
                    total = 0
                    while chunk := source.read(1024 * 1024):
                        total += len(chunk)
                        if total > info.file_size:
                            raise ValueError('Archive exceeds declared size')
                        dest.write(chunk)
                if file_hash(temporary) != expected['sha256']:
                    raise ValueError('CSV hash mismatch: wrong or edited R3 cohort')
                temporary.replace(output)
            finally:
                if temporary.exists():
                    temporary.unlink()
    else:
        if path.suffix.lower() != '.csv' or file_hash(path) != expected['sha256']:
            raise ValueError('CSV hash mismatch: wrong or edited R3 cohort')
        # Already-local CSVs can be large; avoid making an unnecessary second copy.
        output = path
    if file_hash(output) != expected['sha256']:
        raise ValueError('Verified cohort changed while reading')
    print('Verified', cohort, '|', expected['users'], 'users |', expected['rows'], 'rows', flush=True)
    return output

def grouped_csv(path, chunksize=50000):
    """Bounded raw-memory reader for R3's user-contiguous CSVs."""
    pending = None
    seen = set()
    for chunk in pd.read_csv(path, chunksize=chunksize, dtype={'user_id': str, 'transaction_id': str}):
        if pending is not None:
            chunk = pd.concat([pending, chunk], ignore_index=True)
        if chunk.user_id.isna().any():
            raise ValueError('Missing user ID')
        last = chunk.user_id.iloc[-1]
        pending = chunk.loc[chunk.user_id.eq(last)].copy()
        ready = chunk.loc[~chunk.user_id.eq(last)]
        for uid, group in ready.groupby('user_id', sort=False):
            if uid in seen:
                raise ValueError('R3 CSV must keep each user in one contiguous block')
            seen.add(uid)
            yield uid, group.reset_index(drop=True)
    if pending is not None and len(pending):
        uid = pending.user_id.iloc[0]
        if uid in seen:
            raise ValueError('Noncontiguous user records')
        yield uid, pending.reset_index(drop=True)

def cache_key(cohort, manifest, settings):
    return hashlib.sha256(json.dumps(dict(cohort=cohort, hash=manifest['cohorts'][cohort]['sha256'],
        code=FEATURE_CODE_HASH, settings=settings, versions=VERSIONS), sort_keys=True).encode()).hexdigest()

def disjoint_cohorts(*cohorts):
    seen = set()
    for data in cohorts:
        if seen & data['users']:
            raise ValueError('User overlap between cohorts; no fitting/evaluation allowed')
        seen |= data['users']

def later_masks(data, settings):
    left = data['start'] + pd.Timedelta(weeks=settings['train_weeks'])
    return data['f'].time.ge(left), data['a'].time.ge(left)

def event_contained(a, mask):
    total = a.loc[a.event_id.notna()].groupby('event_id').size()
    count = a.loc[mask & a.event_id.notna()].groupby('event_id').size()
    eligible = count.index[count.eq(total.reindex(count.index))]
    return mask & (a.event_id.isna() | a.event_id.isin(eligible))

def make_settings(manifest, mode='full'):
    weeks = manifest['weeks_per_user']
    return dict(train_weeks=int(weeks * .77), forest_fit_weeks=int(weeks * .62), mode=mode,
                max_epochs=2 if mode == 'smoke' else 120, batch_size=256, patience=12,
                trees=24 if mode == 'smoke' else 300, seed=42, anomaly_fpr_budget=.01)

def candidate_forecast(model, x, f):
    return predict_forecast_v5(dict(kind='lstm'), model, x, f)

def save_lstm_fit(fit, folder):
    import joblib
    folder = Path(folder)
    fit['model'].save(folder / 'lstm.keras')
    joblib.dump(fit['scales'], folder / 'scalers.joblib')
    metadata = dict(best_epoch=fit['best_epoch'], history=fit['history'], spec=fit['spec'], bias_correction=0.)
    json_write(folder / 'fit.json', metadata)

def load_lstm_fit(folder):
    # Caller verifies its own run's artifact hashes before deserializing.
    import tensorflow as tf
    import joblib
    folder = Path(folder)
    result = json.loads((folder / 'fit.json').read_text())
    result.update(model=tf.keras.models.load_model(folder / 'lstm.keras', compile=False), scales=joblib.load(folder / 'scalers.joblib'))
    return result

def forecast_report(f, prediction):
    metrics = regression_metrics(f.y, prediction)
    group = f.assign(prediction=prediction)
    # Three chronological blocks expose variation instead of selecting a lucky week.
    times = sorted(group.time.unique())
    blocks = np.array_split(times, 3)
    by_block = [regression_metrics(g.y, g.prediction)['WAPE'] for b in blocks
                if len(g := group.loc[group.time.isin(b)])]
    metrics['worst_block_WAPE'] = float(np.nanmax(by_block))
    metrics['selection_objective'] = (metrics['WAPE'] + metrics['worst_block_WAPE'])/2
    metrics['weeks'] = len(times)
    return metrics

def fit_and_select_lstm(train, validation, settings, run_dir):
    rows, best_fit, best_objective = [], None, np.inf
    fm, _ = later_masks(validation, settings)
    vf, vx = validation['f'].loc[fm], validation['fx'].loc[fm]
    if vf.empty:
        raise ValueError('No future validation weeks')
    for name, pred in [('last_week', vf['last']), ('mean4', vf.mean4), ('recurring_mean', vf.recurring),
                       ('recurring_median', vf.robust_recurring), ('seasonal_lag4', vx.lag4*vf.scale)]:
        rows.append(dict(model=name, **forecast_report(vf, pred)))
    for index, spec in enumerate(r3_lstm_specs(settings['mode'])):
        name = 'lstm_' + str(index)
        folder = Path(run_dir) / name
        complete = folder / 'complete.json'
        print('LSTM candidate', name, spec, flush=True)
        if complete.exists():
            stored = json.loads(complete.read_text())
            if stored['spec'] != spec:
                raise ValueError('Candidate cache specification changed')
            for path, digest in stored['hashes'].items():
                if file_hash(folder / path) != digest:
                    raise ValueError('Candidate artifact changed')
            fit = load_lstm_fit(folder)
        else:
            fit = train_r3_lstm(train['fx'], train['f'], spec, settings, folder)
            save_lstm_fit(fit, folder)
            json_write(complete, dict(spec=spec, hashes={p: file_hash(folder/p) for p in ('lstm.keras', 'scalers.joblib', 'fit.json')}))
        prediction = candidate_forecast(fit, vx, vf)
        measured = forecast_report(vf, prediction)
        rows.append(dict(model=name, spec=json.dumps(spec), best_epoch=fit['best_epoch'], **measured))
        if measured['selection_objective'] < best_objective:
            best_objective, best_fit = measured['selection_objective'], dict(fit, candidate=name)
        pd.DataFrame(rows).to_csv(Path(run_dir) / 'forecast_validation.csv', index=False)
    if best_fit is None:
        raise ValueError('No finite LSTM selection result')
    return best_fit, pd.DataFrame(rows)

def forest_specs_r3():
    return [dict(view=view, samples=samples) for view in ('excess', 'context', 'rhythm_only') for samples in (128,512)]

def forest_score(fitted, x):
    return -fitted['model'].score_samples(np.log1p(x[fitted['columns']]))

def fit_forest(spec, x, settings):
    columns = rhythm_views(x)[spec['view']]
    model = IsolationForest(n_estimators=settings['trees'], max_samples=min(spec['samples'],len(x)),
                            contamination='auto', random_state=settings['seed'], n_jobs=2).fit(np.log1p(x[columns]))
    return dict(model=model, columns=columns, spec=spec)

def calibrate_forest(fitted, a, x, budget):
    score = forest_score(fitted, x)
    threshold = fpr_capped_threshold(a.label, score, float('nan'), budget)
    if a.label.nunique() != 2 or not np.isfinite(threshold['threshold']):
        raise ValueError('Calibration needs explicit normal and anomaly labels; no unsupported fallback')
    return dict(threshold, support_warning=int(a.label.eq(1).sum()) < 30)

def fit_and_select_forests(train, validation, settings, run_dir):
    print('STAGE: ISOLATION FOREST (scikit-learn CPU; GPU is used by LSTM only)',flush=True)
    start = train['start']
    left = start + pd.Timedelta(weeks=settings['forest_fit_weeks'])
    fitmask = event_contained(train['a'], train['a'].time.lt(left))
    calmask = event_contained(train['a'], train['a'].time.ge(left))
    _, vm = later_masks(validation, settings)
    candidates, rows = {}, []
    for index, spec in enumerate(forest_specs_r3()):
        name = 'if_' + str(index)
        fitted = fit_forest(spec, train['ax'].loc[fitmask], settings)
        for budget in (settings['anomaly_fpr_budget'], settings['anomaly_fpr_budget']/2):
            threshold = calibrate_forest(fitted, train['a'].loc[calmask], train['ax'].loc[calmask], budget)
            score = forest_score(fitted, validation['ax'].loc[vm])
            record = dict(model=name, spec=json.dumps(spec), budget=budget,
                          **classification_metrics(validation['a'].loc[vm,'label'], score, threshold['threshold']))
            rows.append(record)
            candidates[(name,budget)] = dict(fitted, threshold=threshold)
    table = pd.DataFrame(rows)
    valid = table.loc[table.budget.eq(settings['anomaly_fpr_budget']) & table.F1.notna()]
    if valid.empty:
        raise ValueError('Validation needs labelled classes for IF selection')
    single_row = valid.sort_values(['F1','precision'], ascending=False, kind='stable').iloc[0]
    half = settings['anomaly_fpr_budget']/2
    # Fixed composition, independently calibrated thresholds. Both scorers are IFs.
    union_parts = [candidates[('if_0',half)], candidates[('if_5',half)]]
    alerts = np.logical_or.reduce([forest_score(part, validation['ax'].loc[vm]) >= part['threshold']['threshold'] for part in union_parts])
    union_metrics = classification_metrics(validation['a'].loc[vm,'label'], alerts.astype(float), .5)
    # Boolean union has no continuous ranking score. Do not report misleading AP/AUC.
    union_metrics.update(AP=np.nan, ROC_AUC=np.nan)
    rows.append(dict(model='two_if_union', spec='excess128 + rhythm_only512', budget=settings['anomaly_fpr_budget'], **union_metrics))
    selected_specs = dict(single=[candidates[(single_row.model, settings['anomaly_fpr_budget'])]['spec']],
                          union=[part['spec'] for part in union_parts])
    winner = 'union' if union_metrics['F1'] > single_row.F1 else 'single'
    pd.DataFrame(rows).to_csv(Path(run_dir) / 'anomaly_validation.csv', index=False)
    return selected_specs, winner, pd.DataFrame(rows)

def freeze_r3(lstm, selected_specs, winner, train, calibration, settings, run_dir, run_identity):
    print('STAGE: FREEZE / IF REFIT / INDEPENDENT CALIBRATION (mostly CPU)',flush=True)
    import joblib
    import shutil
    run_dir=Path(run_dir)
    committed=run_dir/'frozen'
    if (run_dir/'frozen_manifest_digest.json').exists():
        raise ValueError('Models already committed; do not refreeze after evaluation')
    frozen=run_dir/'freeze_staging'
    if frozen.exists():
        shutil.rmtree(frozen)
    frozen.mkdir()
    save_lstm_fit(lstm, frozen)
    _, cm = later_masks(calibration, settings)
    cm = event_contained(calibration['a'], cm)
    if not cm.any():
        raise ValueError('No independent calibration rows')
    models, rows, configurations = [], [], {}
    for architecture, specs in selected_specs.items():
        configurations[architecture] = []
        for index, spec in enumerate(specs):
            fitted = fit_forest(spec, train['ax'], settings)
            threshold = calibrate_forest(fitted, calibration['a'].loc[cm], calibration['ax'].loc[cm], settings['anomaly_fpr_budget']/len(specs))
            name = architecture + '_' + str(index)
            fitted.update(threshold=threshold, name=name)
            models.append(fitted)
            configurations[architecture].append(name)
            score = forest_score(fitted, calibration['ax'].loc[cm])
            rows.append(dict(model=name, **threshold, **classification_metrics(calibration['a'].loc[cm,'label'],score,threshold['threshold'])))
    joblib.dump(models, frozen/'forests.joblib')
    pd.DataFrame(rows).to_csv(Path(run_dir)/'anomaly_calibration.csv',index=False)
    manifest = dict(protocol='v6-r3-kaggle-v1', run_identity=run_identity, settings=settings,
                    code_hash=FEATURE_CODE_HASH, versions=VERSIONS, forecast_candidate=lstm['candidate'],
                    configurations=configurations, selected_anomaly_architecture=winner,
                    anomaly_selection_source='validation only; calibration never selects architecture',
                    calibration_support=int(cm.sum()), candidate_only=True, backend_compatible=False,
                    hashes={p:file_hash(frozen/p) for p in ('lstm.keras','scalers.joblib','fit.json','forests.joblib')})
    json_write(frozen/'manifest.json',manifest)
    # Verify saved/reloaded inference now, before final uploads are permitted.
    loaded = read_frozen_folder(frozen,manifest,run_identity)
    probe_f, probe_x = train['f'].iloc[:32], train['fx'].iloc[:32]
    np.testing.assert_allclose(candidate_forecast(lstm,probe_x,probe_f), candidate_forecast(loaded['lstm'],probe_x,probe_f),atol=.01,rtol=0)
    for a,b in zip(models, loaded['forests']):
        np.testing.assert_allclose(forest_score(a,train['ax'].iloc[:32]),forest_score(b,train['ax'].iloc[:32]),atol=1e-10,rtol=0)
    json_write(frozen/'parity_passed.json',dict(forecast_tolerance_KES=.01,anomaly_tolerance=1e-10))
    if committed.exists():
        shutil.rmtree(committed)
    frozen.replace(committed)
    json_write(run_dir/'frozen_manifest_digest.json',dict(sha256=file_hash(committed/'manifest.json'),parity_verified=True))
    return loaded, pd.DataFrame(rows)

def load_r3_frozen(run_dir, run_identity):
    folder = Path(run_dir)/'frozen'
    commit=json.loads((Path(run_dir)/'frozen_manifest_digest.json').read_text())
    if not commit.get('parity_verified') or not (folder/'parity_passed.json').exists():
        raise ValueError('Freeze is incomplete: reload parity was not committed')
    digest=commit['sha256']
    if file_hash(folder/'manifest.json') != digest:
        raise ValueError('Frozen manifest changed')
    manifest=json.loads((folder/'manifest.json').read_text())
    return read_frozen_folder(folder,manifest,run_identity)

def read_frozen_folder(folder, manifest, run_identity):
    import joblib
    folder=Path(folder)
    if manifest['run_identity'] != run_identity or manifest['code_hash'] != FEATURE_CODE_HASH or manifest['versions'] != VERSIONS:
        raise ValueError('Run configuration or implementation changed after freeze')
    for name, value in manifest['hashes'].items():
        if file_hash(folder/name) != value:
            raise ValueError('Frozen artifact changed: ' + name)
    return dict(manifest=manifest, lstm=load_lstm_fit(folder), forests=joblib.load(folder/'forests.joblib'))

def evaluate_r3(frozen, data, settings, split):
    fm, am = later_masks(data, settings)
    if not fm.any() or not am.any():
        raise ValueError('No future assessment samples')
    f,x,a,ax = data['f'].loc[fm],data['fx'].loc[fm],data['a'].loc[am],data['ax'].loc[am]
    prediction=candidate_forecast(frozen['lstm'],x,f)
    forecasts=[dict(split=split,model='LSTM',**forecast_report(f,prediction))]
    for name,column in [('recurring_median','robust_recurring'),('last_week','last'),('mean4','mean4')]:
        forecasts.append(dict(split=split,model=name,**forecast_report(f,f[column])))
    weekly=[dict(split=split,week=str(week),**regression_metrics(g.y,g.prediction)) for week,g in f.assign(prediction=prediction).groupby('time')]
    anomalies, families=[],[]
    by_name={m['name']:m for m in frozen['forests']}
    for name,parts in frozen['manifest']['configurations'].items():
        models=[by_name[p] for p in parts]
        scores=[forest_score(m,ax) for m in models]
        flags=np.logical_or.reduce([s >= m['threshold']['threshold'] for s,m in zip(scores,models)])
        metrics=classification_metrics(a.label,scores[0],models[0]['threshold']['threshold']) if len(models)==1 else classification_metrics(a.label,flags.astype(float),.5)
        if len(models)>1:
            metrics.update(AP=np.nan,ROC_AUC=np.nan)
        anomalies.append(dict(split=split,model=name,selected=name==frozen['manifest']['selected_anomaly_architecture'],**metrics))
        for family,g in a.assign(alert=flags).loc[a.label.eq(1)].groupby('family'):
            families.append(dict(split=split,model=name,family=family,support=len(g),TP=int(g.alert.sum()),recall=float(g.alert.mean())))
    return dict(forecast=forecasts,anomaly=anomalies,weekly=weekly,family=families)

def final_claim(manifest, state_dir, run_identity):
    """Stable dataset-level ledger across run folders. No new configuration may reset it."""
    ledger=Path(state_dir)/'holdout_ledger'
    ledger.mkdir(parents=True,exist_ok=True)
    identities=[manifest['cohorts'][name]['sha256'] for name in ('test','test_shifted')]
    for identity in identities:
        if (ledger/(identity+'.json')).exists():
            raise RuntimeError('This R3 holdout was already consumed in this state folder; do not rerun it for selection')
    for identity in identities:
        with (ledger/(identity+'.json')).open('x',encoding='utf-8') as handle:
            json.dump(dict(run=run_identity,status='consumed before any test file is loaded'),handle)

def archive_results(run_dir):
    import zipfile
    run_dir=Path(run_dir)
    archive=run_dir.parent/(run_dir.name+'_results.zip')
    with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as zipped:
        for path in sorted(run_dir.rglob('*')):
            if path.is_file() and path.suffix.lower() in ('.csv','.json','.keras','.joblib','.png'):
                zipped.write(path,arcname=path.relative_to(run_dir).as_posix())
    return archive


In [ ]:
# Importable CPU-only module prevents each worker importing TensorFlow/CUDA.
import importlib.util
WORKER_CODE_HASH = '175713ef2d93d959138c0f687823f9383cccad95ac847ca2018cf6d53cab63bd'
worker_source = 'import hashlib, json\nfrom pathlib import Path\nfrom collections import defaultdict, deque\nimport numpy as np\nimport pandas as pd\nCOLUMN_MAP = {}\ndef parse_dates(values):\n    return pd.to_datetime(values, errors=\'raise\', utc=True, format=\'mixed\').dt.tz_convert(\'Africa/Nairobi\').dt.tz_localize(None)\n\ndef clean_data(raw, blank_normal=False):\n    data = raw.rename(columns=COLUMN_MAP).copy()\n    if not data.columns.is_unique:\n        raise ValueError(\'COLUMN_MAP creates duplicate columns\')\n    required_columns = {\'user_id\', \'transaction_timestamp\', \'amount\', \'category\'}\n    missing_columns = required_columns - set(data.columns)\n    if missing_columns:\n        raise ValueError(\'Missing columns: \' + \', \'.join(sorted(missing_columns)))\n    for column in (\'user_id\', \'category\'):\n        if data[column].isna().any() or data[column].astype(str).str.strip().eq(\'\').any():\n            raise ValueError(column + \' contains missing values\')\n        data[column] = data[column].astype(str).str.strip()\n    data[\'transaction_timestamp\'] = parse_dates(data.transaction_timestamp)\n    data[\'amount\'] = pd.to_numeric(data.amount, errors=\'raise\')\n    if data.transaction_timestamp.isna().any() or not np.isfinite(data.amount).all() or data.amount.lt(0).any():\n        raise ValueError(\'Dates must be present and amounts finite and non-negative KES\')\n    if \'transaction_id\' not in data:\n        data[\'transaction_id\'] = [\'row_\' + str(i) for i in range(len(data))]\n    if data.transaction_id.isna().any():\n        raise ValueError(\'transaction_id contains missing values\')\n    data[\'transaction_id\'] = data.transaction_id.astype(str).str.strip()\n    if not data.transaction_id.is_unique or data.transaction_id.eq(\'\').any():\n        raise ValueError(\'transaction_id must be nonblank and unique; resolve duplicate records first\')\n    if \'transaction_type\' not in data:\n        data[\'transaction_type\'] = \'expense\'\n    data[\'transaction_type\'] = data.transaction_type.astype(str).str.strip().str.lower()\n    if not data.transaction_type.isin([\'expense\', \'income\']).all():\n        raise ValueError(\'transaction_type must be expense or income\')\n    labels = data.get(\'is_anomaly\', pd.Series(pd.NA, index=data.index)).astype(\'string\').str.strip().str.lower()\n    mapping = {\'true\': 1., \'1\': 1., \'1.0\': 1., \'yes\': 1., \'false\': 0., \'0\': 0., \'0.0\': 0., \'no\': 0.}\n    if (labels.notna() & labels.ne(\'\') & ~labels.isin(mapping)).any():\n        raise ValueError(\'is_anomaly contains invalid labels; use 0/1, true/false, yes/no or blank\')\n    data[\'label\'] = labels.map(mapping).astype(float)\n    if blank_normal:\n        if \'is_anomaly\' not in data:\n            raise ValueError(\'Cannot infer normal labels when is_anomaly is entirely absent\')\n        data[\'label\'] = data.label.fillna(0.)\n    data[\'merchant\'] = data.get(\'merchant\', pd.Series(\'\', index=data.index)).fillna(\'\').astype(str).str.strip()\n    data[\'event_id\'] = data.get(\'event_id\', pd.Series(pd.NA, index=data.index)).astype(\'string\').str.strip().replace(\'\', pd.NA)\n    coverage = {\'observation_start\', \'observation_end\'} & set(data.columns)\n    if coverage and len(coverage) != 2:\n        raise ValueError(\'Provide both observation_start and exclusive observation_end\')\n    for column in coverage:\n        data[column] = parse_dates(data[column])\n        if data[column].isna().any() or data.groupby(\'user_id\')[column].nunique().ne(1).any():\n            raise ValueError(column + \' must be present and constant per user\')\n    if coverage and ((data.transaction_timestamp < data.observation_start) | (data.transaction_timestamp >= data.observation_end)).any():\n        raise ValueError(\'Transactions fall outside declared observation coverage\')\n    if \'anomaly_type\' in data:\n        data[\'family\'] = data[\'anomaly_type\'].fillna(\'unspecified\').astype(str)\n    expense = data.loc[data.transaction_type.eq(\'expense\')].copy()\n    if expense.empty:\n        raise ValueError(\'No expense records found\')\n    return expense.sort_values([\'user_id\', \'transaction_timestamp\', \'transaction_id\']).reset_index(drop=True)\nA_FEATURES = [\'amount_ratio\', \'category_ratio\', \'category_deviation\', \'merchant_ratio\', \'count_hour\', \'sum_hour_ratio\', \'count_day\', \'sum_week_ratio\', \'duplicate_hour\', \'merchants_hour\', \'night_rarity\', \'category_rarity\', \'user_cold\', \'merchant_missing\']\ndef next_occurrence(schedule, current):\n    if schedule[\'step\'] == \'month\':\n        month = current.normalize().replace(day=1) + pd.DateOffset(months=1)\n        day = min(schedule[\'anchor\'], month.days_in_month)\n        return month.replace(day=day) + (schedule[\'last\'] - schedule[\'last\'].normalize())\n    return current + pd.Timedelta(days=round(schedule[\'period\']))\n\nV5_F_FEATURES = ([f\'lag{i}\' for i in range(1, 9)] +\n                 [f\'nonrec_lag{i}\' for i in range(1, 9)] +\n                 [\'mean4\', \'std8\', \'recurring\', \'nonrecurring\', \'log_scale\',\n                  \'nonrec_median\', \'nonrec_std\', \'due_confidence\', \'count4\', \'zero_fraction\'])\nRHYTHM_FEATURES = [\'day_count_excess\', \'week_count_excess\', \'gap_compression\',\n                   \'recurring_increase\', \'recurring_known\']\n\ndef _spread(values):\n    """Robust MAD-based spread with a one-transaction floor for sparse users."""\n    if not len(values):\n        return 1.\n    array = np.asarray(values, dtype=float)\n    centre = float(np.median(array))\n    return max(float(np.median(np.abs(array - centre))) * 1.4826, 1.)\n\ndef rhythm_features(data):\n    """One forward pass per user; simultaneous transactions share only earlier history."""\n    rows = []\n    for _, group in data.groupby(\'user_id\', sort=False):\n        day_counts, week_counts, gaps = deque(maxlen=90), deque(maxlen=26), deque(maxlen=200)\n        amounts, times = defaultdict(lambda: deque(maxlen=12)), defaultdict(lambda: deque(maxlen=12))\n        recent = deque()\n        current_day = current_week = None\n        day_total = week_total = 0\n        previous = None\n        for t, batch in group.groupby(\'transaction_timestamp\', sort=True):\n            day, week = t.normalize(), t.normalize() - pd.Timedelta(days=t.weekday())\n            if current_day is None:\n                current_day, current_week = day, week\n            # Completed periods only; the in-progress day/week never enters its own baseline.\n            while current_day < day:\n                day_counts.append(day_total)\n                current_day, day_total = current_day + pd.Timedelta(days=1), 0\n            while current_week < week:\n                week_counts.append(week_total)\n                current_week, week_total = current_week + pd.Timedelta(weeks=1), 0\n            while recent and recent[0] < t - pd.Timedelta(days=7):\n                recent.popleft()\n            day_window = sum(1 for r in recent if r >= t - pd.Timedelta(days=1))\n            week_window = len(recent)\n            day_median = float(np.median(day_counts)) if day_counts else 0.\n            week_median = float(np.median(week_counts)) if week_counts else 0.\n            day_excess = max(0., (day_window - day_median) / _spread(day_counts))\n            week_excess = max(0., (week_window - week_median) / _spread(week_counts))\n            gap = (t - previous).total_seconds() / 60 if previous is not None else None\n            compression = 0.\n            if gap is not None and gaps:\n                compression = min(float(np.median(gaps)) / max(gap, 1.), 100.)\n            for row in batch.itertuples():\n                key = (row.category, row.merchant)\n                past_times, past_amounts = times[key], amounts[key]\n                increase, known = 0., 0.\n                if len(past_times) >= 3:\n                    intervals = np.diff(pd.DatetimeIndex(past_times).to_numpy()) / np.timedelta64(1, \'D\')\n                    cadence = float(np.median(intervals))\n                    if 25 <= cadence <= 35 and float(np.median(np.abs(intervals - cadence))) <= .2 * cadence:\n                        expected = float(np.median(past_amounts))\n                        if expected > 0:\n                            known, increase = 1., max(0., row.amount / expected - 1.)\n                rows.append((row.Index, [day_excess, week_excess, compression, increase, known]))\n            for row in batch.itertuples():\n                times[(row.category, row.merchant)].append(t)\n                amounts[(row.category, row.merchant)].append(row.amount)\n                recent.append(t)\n                day_total += 1\n                week_total += 1\n            # Record the gap only after scoring, so it cannot inform its own baseline.\n            if gap is not None:\n                gaps.append(gap)\n            previous = t\n    return pd.DataFrame([values for _, values in rows], index=[index for index, _ in rows],\n                        columns=RHYTHM_FEATURES, dtype=float).reindex(data.index)\ndef json_safe(value):\n    if isinstance(value, dict):\n        return {str(k): json_safe(v) for k, v in value.items()}\n    if isinstance(value, (list, tuple)):\n        return [json_safe(v) for v in value]\n    if isinstance(value, (np.integer, np.bool_)):\n        return value.item()\n    if isinstance(value, (float, np.floating)):\n        return float(value) if np.isfinite(value) else None\n    return value\ndef file_hash(path):\n    digest = hashlib.sha256()\n    with Path(path).open(\'rb\') as handle:\n        for chunk in iter(lambda: handle.read(1024 * 1024), b\'\'):\n            digest.update(chunk)\n    return digest.hexdigest()\n\ndef json_write(path, value):\n    path=Path(path)\n    content=json.dumps(json_safe(value), indent=2, sort_keys=True, allow_nan=False)\n    temporary=path.with_name(path.name+\'.writing\')\n    temporary.write_text(content,encoding=\'utf-8\')\n    temporary.replace(path)\n"""Fast parity-preserving V5 features, embedded in a CPU-only worker module.\n\nNumerical imports, schemas and the reference helpers are provided by the builder.\nThis file never imports TensorFlow or alters labels, lookbacks or cohort membership.\n"""\n\n\ndef fast_schedule(times, amounts):\n    dates = pd.DatetimeIndex(times).unique().sort_values()\n    if len(dates) < 2:\n        return None\n    months = dates.year * 12 + dates.month\n    gaps = np.diff(dates.to_numpy()) / np.timedelta64(1, \'D\')\n    if months.is_unique and dates.day.max() - dates.day.min() <= 3 and np.median(np.diff(months)) <= 2 and np.max(np.diff(months)) <= 3:\n        step, period, anchor = \'month\', 30., int(round(float(np.median(dates.day))))\n    else:\n        period = float(np.median(gaps))\n        if not 5 <= period <= 16 or np.median(np.abs(gaps-period)) > .12*period:\n            return None\n        step, anchor = \'days\', None\n    if len(np.unique(times)) != len(times):\n        # Preserve the exact reference\'s pandas default sort behavior for unequal\n        # tied timestamps at the last-six cutoff. Do not silently change schedules.\n        recent=pd.DataFrame({\'timestamp\':times,\'amount\':amounts}).sort_values(\'timestamp\').amount.tail(6).to_numpy()\n    else:\n        recent = np.asarray(amounts[-6:])\n    amount = float(np.median(recent))\n    if amount <= 0 or float(np.median(np.abs(recent-amount))) > .30*amount:\n        return None\n    return dict(last=dates[-1],amount=amount,step=step,period=period,anchor=anchor,\n                confidence=min(1.,(len(dates)-1)/2))\n\n\ndef forecast_features_fast(data, variant=\'pooled\'):\n    """Pre-group/slice arrays once instead of rebuilding pandas groupbys every week."""\n    rows, values, skipped = [], [], 0\n    for user, group in data.groupby(\'user_id\',sort=True):\n        group = group.sort_values([\'transaction_timestamp\',\'transaction_id\'])\n        start = group.observation_start.iloc[0] if \'observation_start\' in group else group.transaction_timestamp.min().normalize()\n        end = group.observation_end.iloc[0] if \'observation_end\' in group else group.transaction_timestamp.max().normalize()\n        first = start.normalize()-pd.Timedelta(days=start.weekday())\n        if first < start:\n            first += pd.Timedelta(weeks=1)\n        last = end.normalize()-pd.Timedelta(days=end.weekday())\n        boundaries = pd.date_range(first,last,freq=\'7D\')\n        if len(boundaries)<10:\n            skipped += 1\n            continue\n        t=group.transaction_timestamp.to_numpy(dtype=\'datetime64[ns]\')\n        amount=group.amount.to_numpy(dtype=float)\n        weeks=(group.transaction_timestamp.dt.normalize()-pd.to_timedelta(group.transaction_timestamp.dt.weekday,unit=\'D\')).to_numpy(dtype=\'datetime64[ns]\')\n        codes=np.searchsorted(boundaries.to_numpy(),weeks)\n        valid=(codes>=0)&(codes<len(boundaries)-1)&(weeks>=first.to_datetime64())\n        weekly=np.bincount(codes[valid],weights=amount[valid],minlength=len(boundaries)-1)\n        counts=np.bincount(codes[valid],minlength=len(boundaries)-1)\n        categories=[]\n        # Positions correspond to sorted transaction arrays. Labels/IDs never used as inputs.\n        for _, positions in group.reset_index(drop=True).groupby(\'category\',sort=True).indices.items():\n            positions=np.asarray(positions)\n            merchants=[positions[np.flatnonzero(group.merchant.iloc[positions].to_numpy()==merchant)]\n                       for merchant in sorted(group.merchant.iloc[positions].unique())]\n            categories.append((positions,merchants))\n        for i in range(8,len(weekly)):\n            left,right=boundaries[i],boundaries[i+1]\n            lag_start=boundaries[i-8]\n            lower=(left-pd.Timedelta(weeks=26)).to_datetime64()\n            upper=left.to_datetime64()\n            lag=lag_start.to_datetime64()\n            past=weekly[i-8:i]\n            scale=max(float(past.mean()),1.)\n            recurring=np.zeros(len(group),dtype=bool)\n            due,confidence=0.,[]\n            for positions,merchant_groups in categories:\n                cat=positions[(t[positions]>=lower)&(t[positions]<upper)]\n                schedule=fast_schedule(t[cat],amount[cat]) if variant==\'pooled\' else None\n                candidates=[(cat,schedule)] if schedule is not None else []\n                if schedule is None:\n                    for member in merchant_groups:\n                        selected=member[(t[member]>=lower)&(t[member]<upper)]\n                        candidates.append((selected,fast_schedule(t[selected],amount[selected])))\n                for selected,schedule in candidates:\n                    if schedule is None:\n                        continue\n                    date=next_occurrence(schedule,schedule[\'last\'])\n                    if date < left-pd.Timedelta(days=40):\n                        continue\n                    while date < right:\n                        if date >= left:\n                            due += schedule[\'amount\']\n                            confidence.append(schedule[\'confidence\'])\n                        date=next_occurrence(schedule,date)\n                    recurring[selected[t[selected]>=lag]]=True\n            mask=(t>=lag)&(t<upper)&~recurring\n            nonrec=np.bincount(codes[mask]-(i-8),weights=amount[mask],minlength=8)\n            mean=float(nonrec.mean())\n            values.append([*(past[::-1]/scale),*(nonrec[::-1]/scale),past[-4:].mean()/scale,\n                past.std()/scale,due/scale,mean/scale,np.log(scale),np.median(nonrec)/scale,\n                nonrec.std()/scale,float(np.mean(confidence)) if confidence else 0.,float(counts[i-4:i].mean()),np.mean(past==0)])\n            rows.append(dict(user_id=user,time=left,end=right,y=weekly[i],scale=scale,last=past[-1],\n                             mean4=past[-4:].mean(),recurring=due+mean,robust_recurring=due+float(np.median(nonrec)),due=due,nonrec_mean=mean))\n    return pd.DataFrame(rows),pd.DataFrame(values,columns=V5_F_FEATURES),skipped\n\n\ndef anomaly_features_fast(data):\n    """Same 18 V5 features; integer time windows avoid repeated Timedelta/Pandas work."""\n    values=[]\n    for _, group in data.groupby(\'user_id\',sort=False):\n        group=group.sort_values([\'transaction_timestamp\',\'transaction_id\'])\n        history=deque(maxlen=100)\n        categories=defaultdict(lambda:deque(maxlen=40))\n        merchants=defaultdict(lambda:deque(maxlen=20))\n        recent,category_counts,seen,nights=deque(),defaultdict(int),0,0\n        rows=list(group.itertuples())\n        times=group.transaction_timestamp.to_numpy(dtype=\'datetime64[ns]\').astype(\'int64\')\n        boundaries=np.r_[0,np.flatnonzero(np.diff(times))+1,len(times)]\n        hour_ns,day_ns=3600*10**9,86400*10**9\n        for left,right in zip(boundaries[:-1],boundaries[1:]):\n            stamp=times[left]\n            hour_of_day=rows[left].transaction_timestamp.hour\n            while recent and recent[0][0] < stamp-7*day_ns:\n                recent.popleft()\n            hour=[r for r in recent if r[0]>=stamp-hour_ns]\n            day=[r for r in recent if r[0]>=stamp-day_ns]\n            for row in rows[left:right]:\n                a,c,m=row.amount,row.category,row.merchant\n                typical=max(float(np.median(history)) if history else 1.,1.)\n                cat=categories[c]\n                center=max(float(np.median(cat)) if cat else typical,1.)\n                spread=max(float(np.median(np.abs(np.asarray(cat)-center)))*1.4826 if cat else 0.,center*.2,1.)\n                mc=max(float(np.median(merchants[m])) if m and merchants[m] else center,1.)\n                feature=[a/typical,a/center,max(0.,(a-center)/spread),a/mc,len(hour),sum(r[1] for r in hour)/typical,\n                    len(day),sum(r[1] for r in recent)/typical,\n                    sum(bool(m) and r[2]==m and r[3]==c and abs(r[1]-a)<.01 for r in hour),\n                    len({r[2] for r in hour if r[2]}),float(hour_of_day<5 or hour_of_day>=23)*(1-(nights+1)/(seen+2)),\n                    1-category_counts[c]/max(1,seen),float(seen<10),float(not m)]\n                values.append((row.Index,feature))\n            for row in rows[left:right]:\n                history.append(row.amount); categories[row.category].append(row.amount)\n                if row.merchant:\n                    merchants[row.merchant].append(row.amount)\n                recent.append((stamp,row.amount,row.merchant,row.category))\n                seen+=1; nights+=int(hour_of_day<5 or hour_of_day>=23); category_counts[row.category]+=1\n    x=pd.DataFrame([v for _,v in values],index=[i for i,_ in values],columns=A_FEATURES,dtype=float).reindex(data.index)\n    x[\'category_excess\']=np.maximum(0.,x.category_ratio-1.)\n    x[\'merchant_excess\']=np.maximum(0.,x.merchant_ratio-1.)\n    x[\'rare_large\']=x.category_rarity*np.log1p(x.category_excess)*(1-x.user_cold)\n    x[\'burst_amount\']=x.count_hour*np.log1p(x.amount_ratio)\n    return x\n\n\ndef worker_shard(uid, raw, cohort, manifest, settings, cache_root, key):\n    """Runs in a CPU-only joblib worker. No TensorFlow/device object is serialized."""\n    import time\n    begin=time.perf_counter()\n    start=parse_dates(raw.observation_start).iloc[0]\n    end=parse_dates(raw.observation_end).iloc[0]\n    if raw.observation_start.nunique()!=1 or raw.observation_end.nunique()!=1 or end-start!=pd.Timedelta(weeks=manifest[\'weeks_per_user\']):\n        raise ValueError(\'Invalid observation coverage\')\n    folder=Path(cache_root)/key/hashlib.sha256(uid.encode()).hexdigest()\n    folder.mkdir(parents=True,exist_ok=True)\n    marker=folder/\'complete.json\'\n    reused=marker.exists()\n    if reused:\n        saved=json.loads(marker.read_text())\n        if saved[\'user\']!=uid or saved[\'cache_key\']!=key:\n            raise ValueError(\'Feature cache identity mismatch\')\n        for name,digest in saved[\'hashes\'].items():\n            if file_hash(folder/name)!=digest:\n                raise ValueError(\'Corrupt feature cache; choose a new cache directory\')\n    else:\n        data=clean_data(raw,False)\n        if data.label.isna().any():\n            raise ValueError(\'Explicit R3 labels required\')\n        data[\'family\']=data.anomaly_type.fillna(\'normal\')\n        if cohort==\'train\':\n            cut=start+pd.Timedelta(weeks=settings[\'train_weeks\'])\n            data=data.loc[data.transaction_timestamp.lt(cut)].copy().reset_index(drop=True)\n            data[\'observation_end\']=cut\n        f,fx,_=forecast_features_fast(data)\n        if f.empty:\n            raise ValueError(\'No forecast windows\')\n        ax=anomaly_features_fast(data).join(rhythm_features(data))\n        a=data[[\'user_id\',\'transaction_id\',\'transaction_timestamp\',\'label\',\'event_id\',\'family\']].rename(columns={\'transaction_timestamp\':\'time\'})\n        f.to_csv(folder/\'forecast.csv\',index=False)\n        a.to_csv(folder/\'anomaly.csv\',index=False)\n        np.savez_compressed(folder/\'features.npz\',fx=fx.to_numpy(),ax=ax.to_numpy(),\n            fx_columns=np.asarray(fx.columns,dtype=str),ax_columns=np.asarray(ax.columns,dtype=str))\n        json_write(marker,dict(user=uid,cache_key=key,hashes={n:file_hash(folder/n) for n in (\'forecast.csv\',\'anomaly.csv\',\'features.npz\')}))\n    return dict(user=uid,start=str(start),rows=len(raw),folder=str(folder),reused=reused,seconds=time.perf_counter()-begin)\n'
worker_dir=WORK_DIR/'worker_source'
worker_dir.mkdir(exist_ok=True)
module_name='spendly_v6_worker_'+WORKER_CODE_HASH[:16]
worker_path=worker_dir/(module_name+'.py')
worker_path.write_text(worker_source,encoding='utf-8')
if str(worker_dir) not in sys.path:
    sys.path.insert(0,str(worker_dir))
V6_WORKER=__import__(module_name)


In [ ]:
"""Bounded CPU worker orchestration; run in notebook main process, no model changes."""


def v6_final_claim(manifest,state_dir,run_identity,previous_state_dirs=()):
    """New run folders must not silently reset a predecessor's consumed data ledger."""
    identities={manifest['cohorts'][name]['sha256'] for name in ('test','test_shifted')}
    roots={Path(state_dir),*(Path(p) for p in previous_state_dirs)}
    for root in roots:
        if any((root/'holdout_ledger'/(identity+'.json')).exists() for identity in identities):
            raise RuntimeError('R3 holdout already consumed in current or predecessor workspace; no final replay allowed')
    return ORIGINAL_FINAL_CLAIM(manifest,state_dir,run_identity)


def cached_features_v6(csv_path,cohort,manifest,cache_root,settings):
    from joblib import Parallel, delayed, parallel_config
    import time
    if file_hash(csv_path)!=manifest['cohorts'][cohort]['sha256']:
        raise ValueError('Cohort CSV changed before feature preparation')
    workers=int(settings.get('feature_workers',2))
    if not 1<=workers<=4:
        raise ValueError('Use 1 to 4 feature workers to bound RAM')
    key=hashlib.sha256(json.dumps(dict(cohort=cohort,csv=manifest['cohorts'][cohort]['sha256'],
        worker=WORKER_CODE_HASH,cut=settings['train_weeks'] if cohort=='train' else None,
        versions={k:VERSIONS[k] for k in ('numpy','pandas')}),sort_keys=True).encode()).hexdigest()
    print('V6 STAGE: optimized CPU features:',cohort,'workers:',workers,
          '| GPU is for LSTM epochs, not pandas.',flush=True)
    begun=time.perf_counter()
    frames,xframes,anomalies,axframes=[],[],[],[]
    ids=set(); raw_count=0; origin=None; timings=[]
    def tasks():
        seen=set()
        for uid,raw in grouped_csv(csv_path):
            if uid in seen:
                raise ValueError('Repeated user block')
            seen.add(uid)
            yield delayed(V6_WORKER.worker_shard)(uid,raw,cohort,manifest,settings,str(cache_root),key)
    with parallel_config(backend='loky',n_jobs=workers,inner_max_num_threads=1):
        completed=Parallel(return_as='generator',pre_dispatch=workers,batch_size=1)(tasks())
        for index,result in enumerate(completed):
            if result['user'] in ids:
                raise ValueError('Duplicate feature user')
            ids.add(result['user']); raw_count+=result['rows']
            start=pd.Timestamp(result['start'])
            if origin is not None and start!=origin:
                raise ValueError('Mismatched cohort observation starts')
            origin=start
            folder=Path(result['folder'])
            f=pd.read_csv(folder/'forecast.csv',dtype={'user_id':str},parse_dates=['time','end'])
            a=pd.read_csv(folder/'anomaly.csv',dtype={'user_id':str,'transaction_id':str,'event_id':'string'},parse_dates=['time'])
            with np.load(folder/'features.npz',allow_pickle=False) as arrays:
                fx=pd.DataFrame(arrays['fx'],columns=arrays['fx_columns'].tolist())
                ax=pd.DataFrame(arrays['ax'],columns=arrays['ax_columns'].tolist())
            expected=A_FEATURES+['category_excess','merchant_excess','rare_large','burst_amount']+RHYTHM_FEATURES
            if list(fx.columns)!=V5_F_FEATURES or list(ax.columns)!=expected or len(f)!=len(fx) or len(a)!=len(ax):
                raise ValueError('Worker output feature contract mismatch')
            if not np.isfinite(fx.to_numpy()).all() or not np.isfinite(ax.to_numpy()).all():
                raise ValueError('Nonfinite worker features')
            frames.append(f);xframes.append(fx);anomalies.append(a);axframes.append(ax)
            timings.append(dict(reused=result['reused'],seconds=result['seconds']))
            if index==0 or (index+1)%10==0:
                elapsed=time.perf_counter()-begun
                remain=manifest['cohorts'][cohort]['users']-index-1
                print(cohort,index+1,'/',manifest['cohorts'][cohort]['users'],
                      '| elapsed min',round(elapsed/60,1),'estimated remaining min',round(elapsed/(index+1)*remain/60,1),flush=True)
    if len(ids)!=manifest['cohorts'][cohort]['users'] or raw_count!=manifest['cohorts'][cohort]['rows']:
        raise ValueError('Cohort counts do not match manifest')
    timing=dict(cohort=cohort,workers=workers,elapsed_seconds=time.perf_counter()-begun,users=len(ids),
                cached_users=sum(t['reused'] for t in timings),raw_rows=raw_count,cache_key=key)
    json_write(Path(cache_root)/key/'timing.json',timing)
    return dict(f=pd.concat(frames,ignore_index=True),fx=pd.concat(xframes,ignore_index=True),
                a=pd.concat(anomalies,ignore_index=True),ax=pd.concat(axframes,ignore_index=True),
                users=ids,start=origin,key=key,timing=timing)


In [ ]:
"""V6 single-T4 training: fixed-LR control versus plateau reduction with refit replay."""


def v6_specs(mode):
    if mode=='smoke':
        return [dict(units=8,learning_rate=.0003,regularization=.001,schedule='plateau')]
    return [dict(units=32,learning_rate=.0003,regularization=.001,schedule='fixed'),
            dict(units=32,learning_rate=.0003,regularization=.001,schedule='plateau'),
            dict(units=64,learning_rate=.0003,regularization=.001,schedule='plateau')]


def v6_lr_recorder(tf):
    class LRHistory(tf.keras.callbacks.Callback):
        def __init__(self):
            super().__init__()
            self.rates=[]
        def on_epoch_begin(self,epoch,logs=None):
            self.rates.append(float(tf.keras.backend.get_value(self.model.optimizer.learning_rate)))
        def on_epoch_end(self,epoch,logs=None):
            if logs is not None:
                logs['learning_rate_used']=self.rates[-1]
    return LRHistory()


def v6_rate_schedule(rates):
    values=tuple(float(v) for v in rates)
    if not values or not np.isfinite(values).all() or min(values)<=0:
        raise ValueError('Finite positive refit learning rates required')
    def schedule(epoch,current=None):
        if not 0<=epoch<len(values):
            raise ValueError('Refit epoch outside recorded schedule')
        return values[epoch]
    return schedule


def train_v6(x,frame,spec,settings,output):
    import tensorflow as tf
    if spec.get('schedule') not in ('fixed','plateau'):
        raise ValueError('Unknown learning-rate schedule')
    dates=sorted(frame.time.unique())
    cutoff=dates[max(1,int(.8*len(dates)))]
    inner=frame.end.le(cutoff)&frame.time.lt(cutoff)
    stopping=frame.time.ge(cutoff)
    if not inner.any() or not stopping.any():
        raise ValueError('Empty chronological stopping slice')
    target=((frame.y-frame.recurring)/frame.scale).to_numpy(dtype='float32')
    weights=(frame.scale/frame.loc[inner,'scale'].mean()).to_numpy(dtype='float32')
    output=Path(output); output.mkdir(parents=True,exist_ok=True)

    def create(mask,validation=None):
        tf.keras.backend.clear_session()
        tf.keras.utils.set_random_seed(settings['seed'])
        scalers=_lstm_scalers_v5(x.loc[mask])
        model,runtime=make_accelerated_lstm(tf,spec,len(V5_F_FEATURES)-16,settings.get('require_gpu',True))
        model.compile(optimizer=tf.keras.optimizers.Adam(spec['learning_rate'],clipnorm=1.),loss='mae',
                      weighted_metrics=[tf.keras.metrics.MeanAbsoluteError(name='weighted_mae')],jit_compile=False)
        json_write(output/'device.json',runtime)
        print('V6 LSTM device:',runtime['training_device'],'cuDNN:',runtime['cudnn_mode'],flush=True)
        options=tf.data.Options(); options.threading.private_threadpool_size=2
        def dataset(m,shuffle=False):
            ds=tf.data.Dataset.from_tensor_slices((_lstm_inputs_v5(x.loc[m],scalers),target[m],weights[m]))
            if shuffle:
                ds=ds.shuffle(int(m.sum()),seed=settings['seed'],reshuffle_each_iteration=True)
            return ds.batch(settings['batch_size']).with_options(options).prefetch(tf.data.AUTOTUNE)
        return model,scalers,dataset(mask,True),dataset(validation) if validation is not None else None

    recorder=v6_lr_recorder(tf)
    callbacks=[recorder,tf.keras.callbacks.TerminateOnNaN()]
    if spec['schedule']=='plateau':
        callbacks.append(tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss',factor=.5,patience=4,
                          min_delta=1e-4,cooldown=1,min_lr=1e-5,verbose=1))
    callbacks += [tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=16,restore_best_weights=True),
                  tf.keras.callbacks.CSVLogger(str(output/'learning_curve.csv')),
                  tf.keras.callbacks.ModelCheckpoint(str(output/'inner_best.keras'),monitor='val_loss',save_best_only=True)]
    model,_,train,stop=create(inner,stopping)
    history=model.fit(train,validation_data=stop,epochs=settings['max_epochs'],verbose=2,callbacks=callbacks)
    if not np.isfinite(history.history['loss']).all() or not np.isfinite(history.history['val_loss']).all():
        raise FloatingPointError('Nonfinite training/stopping loss; candidate not saved')
    best=int(np.argmin(history.history['val_loss'])+1)
    rates=recorder.rates[:best]
    json_write(output/'refit_schedule.json',dict(best_epoch=best,rates=rates,source='inner training chronology only'))
    model,scalers,train,_=create(pd.Series(True,index=frame.index))
    refit=model.fit(train,epochs=best,verbose=2,callbacks=[tf.keras.callbacks.LearningRateScheduler(v6_rate_schedule(rates)),
        tf.keras.callbacks.TerminateOnNaN(),tf.keras.callbacks.CSVLogger(str(output/'refit_learning_curve.csv'))])
    if not np.isfinite(refit.history['loss']).all():
        raise FloatingPointError('Nonfinite refit loss; candidate not saved')
    return dict(model=model,scales=scalers,best_epoch=best,history=history.history,bias_correction=0.,spec=spec)

train_r3_lstm=train_v6
r3_lstm_specs=v6_specs
cached_features=cached_features_v6
ORIGINAL_FINAL_CLAIM=final_claim


## Attach 1/4 — dataset manifest
Choose `datasets\spendly_synthetic_r3\manifest.json` on your computer.
This is the dataset manifest, not `final_report.json` or a model manifest.


In [ ]:
manifest_file=request_one_file('ATTACH/LOCATE 1/4: choose R3 manifest.json','manifest.json',MANIFEST_PATH)
DATASET=read_r3_manifest(manifest_file)
SETTINGS=make_settings(DATASET,MODE)
if not isinstance(GPU_BATCH_SIZE,int) or isinstance(GPU_BATCH_SIZE,bool) or not 16 <= GPU_BATCH_SIZE <= 2048:
    raise ValueError('GPU_BATCH_SIZE must be an integer between 16 and 2048')
if not isinstance(FEATURE_WORKERS,int) or not 1<=FEATURE_WORKERS<=4:
    raise ValueError('FEATURE_WORKERS must be 1,2,3 or 4; reduce to 1 if RAM is limited')
SETTINGS.update(require_gpu=REQUIRE_GPU,training_device=RUNTIME['training_device'],batch_size=GPU_BATCH_SIZE,
    feature_workers=FEATURE_WORKERS,optimizer='Adam_clipnorm1',training_protocol='v6_lr_replay',patience=16)
RUN_ID=hashlib.sha256(json.dumps(dict(dataset={k:v['sha256'] for k,v in DATASET['cohorts'].items()},
    settings=SETTINGS,code=FEATURE_CODE_HASH,versions=VERSIONS),sort_keys=True).encode()).hexdigest()[:24]
RUN_DIR=WORK_DIR/('run_'+RUN_ID)
RUN_DIR.mkdir(exist_ok=True)
json_write(RUN_DIR/'dataset_manifest.json', DATASET)
json_write(RUN_DIR/'run_config.json',dict(run_id=RUN_ID,settings=SETTINGS,code_hash=FEATURE_CODE_HASH,versions=VERSIONS))
json_write(RUN_DIR/'runtime_device.json',RUNTIME)
print('Run:',RUN_ID,'| training weeks:',SETTINGS['train_weeks'],'| max epochs:',SETTINGS['max_epochs'])
print('Full cohort sizes:',{k:v['users'] for k,v in DATASET['cohorts'].items()})
COHORT_FILES={}


## Attach 2/4 — train
Choose `datasets/spendly_synthetic_r3/development/train.zip` (or `train.csv`).


In [ ]:
path=request_one_file('ATTACH/LOCATE 2/4: choose train.zip or train.csv','train.csv',COHORT_PATHS['train'])
COHORT_FILES['train']=verify_cohort_file(path,'train',DATASET,WORK_DIR/'verified_csv')


## Attach 3/4 — validation
Choose `datasets/spendly_synthetic_r3/development/validation.zip` (or `validation.csv`).


In [ ]:
path=request_one_file('ATTACH/LOCATE 3/4: choose validation.zip or validation.csv','validation.csv',COHORT_PATHS['validation'])
COHORT_FILES['validation']=verify_cohort_file(path,'validation',DATASET,WORK_DIR/'verified_csv')


## Attach 4/4 — calibration
Choose `datasets/spendly_synthetic_r3/development/calibration.zip` (or `calibration.csv`).


In [ ]:
path=request_one_file('ATTACH/LOCATE 4/4: choose calibration.zip or calibration.csv','calibration.csv',COHORT_PATHS['calibration'])
COHORT_FILES['calibration']=verify_cohort_file(path,'calibration',DATASET,WORK_DIR/'verified_csv')


## Full-cohort training and independent threshold calibration

Training users are fitted through 77% of covered weeks (week 120 in R3); the last
20% of those training weeks internally selects epoch counts. Validation users are
evaluated after week 120. Each LSTM candidate is refitted on all available training
windows, never validation users. Selection balances overall and worst time-block WAPE.

IF configurations fit early training-user data and use a separate later training
slice for threshold selection, then rank on validation users. After selection, IFs
refit on training data and thresholds are fixed on **calibration users**, after week
120. Single IF and fixed two-IF union are both reported. The validation winner stays
selected even if calibration/test results disagree. Union AP/AUC are intentionally
undefined because a Boolean union is not a continuous ranking score.

Cache files contain synthetic per-user feature data; they are excluded from the
results download. Cache/model hashes catch corruption. Do not restore joblib/Keras
model files from an untrusted source. The code loads only artifacts written by this
run; checksums are not protection against a malicious author able to forge all files.

V6 changes learning-rate scheduling; compare fixed versus plateau candidates on validation, not training loss alone.


In [ ]:
if (RUN_DIR/'frozen_manifest_digest.json').exists():
    print('Models already frozen. Skipping fitting: use summary/final cells; do not retune after viewing final data.')
    FROZEN=load_r3_frozen(RUN_DIR,RUN_ID)
else:
    TRAIN=cached_features(COHORT_FILES['train'],'train',DATASET,WORK_DIR/'feature_cache',SETTINGS)
    VALIDATION=cached_features(COHORT_FILES['validation'],'validation',DATASET,WORK_DIR/'feature_cache',SETTINGS)
    CALIBRATION=cached_features(COHORT_FILES['calibration'],'calibration',DATASET,WORK_DIR/'feature_cache',SETTINGS)
    disjoint_cohorts(TRAIN,VALIDATION,CALIBRATION)
    json_write(RUN_DIR/'preprocessing_timing.json',{k:v['timing'] for k,v in [('train',TRAIN),('validation',VALIDATION),('calibration',CALIBRATION)]})
    print('Feature row counts:',{k:(len(v['f']),len(v['a'])) for k,v in [('train',TRAIN),('validation',VALIDATION),('calibration',CALIBRATION)]})
    LSTM,FORECAST_VALIDATION=fit_and_select_lstm(TRAIN,VALIDATION,SETTINGS,RUN_DIR)
    IF_SPECS,IF_WINNER,ANOMALY_VALIDATION=fit_and_select_forests(TRAIN,VALIDATION,SETTINGS,RUN_DIR)
    print('Selected LSTM:',LSTM['candidate'],'| anomaly architecture:',IF_WINNER)
    FROZEN,CALIBRATION_REPORT=freeze_r3(LSTM,IF_SPECS,IF_WINNER,TRAIN,CALIBRATION,SETTINGS,RUN_DIR,RUN_ID)
    print('Saved/reloaded prediction parity passed. Models and thresholds are frozen.')


In [ ]:
for filename in ('forecast_validation.csv','anomaly_validation.csv','anomaly_calibration.csv'):
    print(filename)
    print(pd.read_csv(RUN_DIR/filename).round(4).to_string(index=False))
for curve in sorted(RUN_DIR.glob('lstm_*/learning_curve.csv')):
    table=pd.read_csv(curve)
    fig,ax=plt.subplots(figsize=(7,3))
    table[['loss','val_loss']].plot(ax=ax)
    ax.set(title=curve.parent.name+' inner-training loss (not external accuracy)',xlabel='Epoch',ylabel='Weighted MAE + regularization')
    fig.tight_layout(); fig.savefig(curve.parent/'learning_curve.png',dpi=120); plt.show(); plt.close(fig)
json_write(RUN_DIR/'development_status.json',dict(stage='frozen_before_test',mode=MODE,
    quality_claim='selection/calibration evidence only; held-out and real-world performance unestablished',
    dataset_hashes={k:v['sha256'] for k,v in DATASET['cohorts'].items()},frozen=FROZEN['manifest']))


## Rule-based recommendation demonstration
The embedded engine below is the existing application implementation. The demonstration
uses synthetic budget/alert context; it is not a new learned model or evidence of
real financial usefulness. Unknown income/budget is not invented from diagnostic profiles.


In [ ]:
"""Typed inputs and outputs for the recommendation engine."""

from dataclasses import asdict, dataclass, field

from typing import Any, Literal, Mapping

Severity = Literal["info", "positive", "warning", "high", "critical"]

def _optional_nonnegative(name: str, value: float | None) -> float | None:
    if value is None:
        return None
    numeric = float(value)
    if numeric < 0:
        raise ValueError(f"{name} cannot be negative.")
    return numeric

@dataclass(frozen=True)
class RecommendationContext:
    forecasted_spending: float | None = None
    budget_amount: float | None = None
    historical_spending_average: float | None = None
    spending_growth_rate: float | None = None
    unusual_spending_detected: bool | None = None
    category_increases: Mapping[str, float] = field(default_factory=dict)
    transaction_frequency_change: float | None = None
    recurring_expense_current: float | None = None
    recurring_expense_previous: float | None = None
    period_income: float | None = None
    period_expenses: float | None = None
    history_periods: int = 0

    def __post_init__(self) -> None:
        for name in (
            "forecasted_spending",
            "budget_amount",
            "historical_spending_average",
            "recurring_expense_current",
            "recurring_expense_previous",
            "period_income",
            "period_expenses",
        ):
            _optional_nonnegative(name, getattr(self, name))
        if self.history_periods < 0:
            raise ValueError("history_periods cannot be negative.")
        for category, increase in self.category_increases.items():
            if not str(category).strip():
                raise ValueError("Category names cannot be blank.")
            float(increase)

@dataclass(frozen=True)
class Recommendation:
    recommendation_code: str
    title: str
    message: str
    severity: Severity
    reason: str
    supporting_values: Mapping[str, Any]
    suggested_action: str
    disclaimer: str
    priority: int

    def to_dict(self) -> dict[str, Any]:
        return asdict(self)

"""Shared, plain-language explanation helpers."""

DISCLAIMER = (
    "This recommendation is automated financial decision support, not "
    "professional financial advice or a guaranteed outcome."
)

def kes(value: float) -> str:
    return f"KES {value:,.2f}"

def percent(value: float) -> str:
    return f"{value * 100:.1f}%"

"""Individual deterministic recommendation rules."""

import re

from collections.abc import Callable

RuleFunction = Callable[[RecommendationContext], Recommendation | None]

def insufficient_history(
    context: RecommendationContext,
) -> Recommendation | None:
    minimum_periods = 8
    if context.history_periods >= minimum_periods:
        return None
    return Recommendation(
        recommendation_code="INSUFFICIENT_HISTORY",
        title="More spending history is needed",
        message=(
            f"Only {context.history_periods} completed period(s) are available. "
            f"At least {minimum_periods} are needed for personalised forecasting."
        ),
        severity="info",
        reason="The forecasting model requires a complete look-back window.",
        supporting_values={
            "available_periods": context.history_periods,
            "required_periods": minimum_periods,
        },
        suggested_action=(
            "Continue adding transactions until a complete history window is available."
        ),
        disclaimer=DISCLAIMER,
        priority=100,
    )

def spending_exceeds_income(
    context: RecommendationContext,
) -> Recommendation | None:
    if context.period_income is None or context.period_expenses is None:
        return None
    if context.period_expenses <= context.period_income:
        return None
    difference = context.period_expenses - context.period_income
    return Recommendation(
        recommendation_code="SPENDING_EXCEEDS_INCOME",
        title="Expenses are above income",
        message=(
            f"Recorded expenses exceed income by {kes(difference)} for this period."
        ),
        severity="critical",
        reason="Period expenses are greater than period income.",
        supporting_values={
            "period_income": context.period_income,
            "period_expenses": context.period_expenses,
            "difference": difference,
        },
        suggested_action=(
            "Review non-essential expenses and adjust the next budget before adding "
            "new discretionary commitments."
        ),
        disclaimer=DISCLAIMER,
        priority=95,
    )

def unusual_spending(
    context: RecommendationContext,
) -> Recommendation | None:
    if context.unusual_spending_detected is not True:
        return None
    return Recommendation(
        recommendation_code="UNUSUAL_SPENDING_REVIEW",
        title="Review unusual spending",
        message=(
            "A recent transaction pattern differs from the available spending history."
        ),
        severity="high",
        reason="The unusual-spending model crossed its configured alert threshold.",
        supporting_values={"unusual_spending_detected": True},
        suggested_action=(
            "Review the flagged transactions and correct any entry that is inaccurate."
        ),
        disclaimer=DISCLAIMER,
        priority=90,
    )

def forecast_exceeds_budget(
    context: RecommendationContext,
) -> Recommendation | None:
    if context.forecasted_spending is None or context.budget_amount is None:
        return None
    if context.forecasted_spending <= context.budget_amount:
        return None
    difference = context.forecasted_spending - context.budget_amount
    return Recommendation(
        recommendation_code="FORECAST_EXCEEDS_BUDGET",
        title="Forecast is above budget",
        message=(
            f"Next-period spending is forecast at "
            f"{kes(context.forecasted_spending)}, which is {kes(difference)} "
            "above the current budget."
        ),
        severity="high",
        reason="Forecasted spending is greater than the configured budget.",
        supporting_values={
            "forecasted_spending": context.forecasted_spending,
            "budget_amount": context.budget_amount,
            "difference": difference,
        },
        suggested_action=(
            "Review the largest planned categories and set a realistic spending limit."
        ),
        disclaimer=DISCLAIMER,
        priority=85,
    )

def rapid_category_increase(
    context: RecommendationContext,
) -> Recommendation | None:
    if not context.category_increases:
        return None
    category, increase = max(
        (
            (str(category).strip(), float(value))
            for category, value in context.category_increases.items()
        ),
        key=lambda item: item[1],
    )
    if increase < 0.30:
        return None
    safe_category = re.sub(r"[^A-Z0-9]+", "_", category.upper()).strip("_")
    return Recommendation(
        recommendation_code=f"CATEGORY_INCREASE_{safe_category}",
        title=f"{category.title()} spending is increasing",
        message=(
            f"{category.title()} spending increased by {percent(increase)} "
            "relative to its comparison period."
        ),
        severity="warning",
        reason="The largest category increase is at or above 30%.",
        supporting_values={
            "category": category,
            "category_increase": increase,
        },
        suggested_action=(
            f"Review recent {category.lower()} transactions and decide whether "
            "the increase should continue."
        ),
        disclaimer=DISCLAIMER,
        priority=75,
    )

def high_transaction_frequency(
    context: RecommendationContext,
) -> Recommendation | None:
    change = context.transaction_frequency_change
    if change is None or change < 0.50:
        return None
    return Recommendation(
        recommendation_code="HIGH_TRANSACTION_FREQUENCY",
        title="Transaction frequency increased",
        message=(
            f"Transaction frequency increased by {percent(change)} compared with "
            "the previous period."
        ),
        severity="warning",
        reason="Transaction frequency increased by at least 50%.",
        supporting_values={"transaction_frequency_change": change},
        suggested_action=(
            "Check whether frequent small purchases are contributing to overspending."
        ),
        disclaimer=DISCLAIMER,
        priority=70,
    )

def recurring_expense_increase(
    context: RecommendationContext,
) -> Recommendation | None:
    current = context.recurring_expense_current
    previous = context.recurring_expense_previous
    if current is None or previous is None or previous <= 0:
        return None
    increase = current / previous - 1
    if increase < 0.10:
        return None
    return Recommendation(
        recommendation_code="RECURRING_EXPENSE_INCREASE",
        title="Recurring expenses increased",
        message=(
            f"Recurring expenses increased by {percent(increase)}, from "
            f"{kes(previous)} to {kes(current)}."
        ),
        severity="warning",
        reason="Recurring expenses increased by at least 10%.",
        supporting_values={
            "recurring_expense_current": current,
            "recurring_expense_previous": previous,
            "increase": increase,
        },
        suggested_action=(
            "Review subscriptions and other recurring charges for changes or "
            "services that are no longer needed."
        ),
        disclaimer=DISCLAIMER,
        priority=65,
    )

def spending_growth(
    context: RecommendationContext,
) -> Recommendation | None:
    growth = context.spending_growth_rate
    if growth is None or growth < 0.25:
        return None
    return Recommendation(
        recommendation_code="SPENDING_GROWTH",
        title="Overall spending is growing",
        message=(
            f"Overall spending increased by {percent(growth)} relative to the "
            "comparison period."
        ),
        severity="warning",
        reason="The spending growth rate is at or above 25%.",
        supporting_values={"spending_growth_rate": growth},
        suggested_action=(
            "Compare the main category totals with the previous period and "
            "prioritise reductions that will not affect essential needs."
        ),
        disclaimer=DISCLAIMER,
        priority=60,
    )

def stable_within_budget(
    context: RecommendationContext,
) -> Recommendation | None:
    forecast = context.forecasted_spending
    budget = context.budget_amount
    historical = context.historical_spending_average
    if forecast is None or budget is None or historical is None:
        return None
    stable_limit = historical * 1.10
    if forecast > budget or forecast > stable_limit:
        return None
    return Recommendation(
        recommendation_code="STABLE_WITHIN_BUDGET",
        title="Spending is on track",
        message=(
            f"The {kes(forecast)} forecast is within budget and close to the "
            "available historical average."
        ),
        severity="positive",
        reason=(
            "Forecasted spending is within budget and no more than 10% above "
            "the historical average."
        ),
        supporting_values={
            "forecasted_spending": forecast,
            "budget_amount": budget,
            "historical_spending_average": historical,
        },
        suggested_action=(
            "Continue monitoring transactions and keep the current budget under review."
        ),
        disclaimer=DISCLAIMER,
        priority=10,
    )

RULES: tuple[RuleFunction, ...] = (
    spending_exceeds_income,
    unusual_spending,
    forecast_exceeds_budget,
    rapid_category_increase,
    high_transaction_frequency,
    recurring_expense_increase,
    spending_growth,
    stable_within_budget,
)

"""Recommendation rule orchestration, priority, and suppression."""

from collections.abc import Iterable

class RecommendationEngine:
    def __init__(
        self,
        rules: Iterable[RuleFunction] = RULES,
        *,
        maximum_recommendations: int = 3,
    ) -> None:
        if maximum_recommendations <= 0:
            raise ValueError("maximum_recommendations must be positive.")
        self._rules = tuple(rules)
        self._maximum_recommendations = maximum_recommendations

    def evaluate(
        self, context: RecommendationContext
    ) -> list[Recommendation]:
        history_result = insufficient_history(context)
        if history_result is not None:
            return [history_result]
        candidates = [
            recommendation
            for rule in self._rules
            if (recommendation := rule(context)) is not None
        ]
        if any(
            recommendation.severity in {"warning", "high", "critical"}
            for recommendation in candidates
        ):
            candidates = [
                recommendation
                for recommendation in candidates
                if recommendation.recommendation_code != "STABLE_WITHIN_BUDGET"
            ]
        by_code: dict[str, Recommendation] = {}
        for recommendation in candidates:
            current = by_code.get(recommendation.recommendation_code)
            if current is None or recommendation.priority > current.priority:
                by_code[recommendation.recommendation_code] = recommendation
        return sorted(
            by_code.values(),
            key=lambda recommendation: (
                -recommendation.priority,
                recommendation.recommendation_code,
            ),
        )[: self._maximum_recommendations]


In [ ]:
demo=RecommendationContext(history_periods=12,forecasted_spending=6000.,budget_amount=5000.,unusual_spending_detected=True)
print('Synthetic example:',[r.to_dict() for r in RecommendationEngine().evaluate(demo)])


In [ ]:
archive=archive_results(RUN_DIR)
print('DEVELOPMENT RESULTS:',archive)
print('Save Version, verify Output files, and download this ZIP from Kaggle Output.')


## Optional final test — deliberately separate from default Run all

**Recommended first:** send the development results ZIP for review before consuming
your holdouts. Do not test repeatedly while picking settings.

When the method is settled, set the confirmation string in the next cell to
`EVALUATE BOTH FROZEN R3 HOLDOUTS ONCE` and run that cell. It reloads the verified
saved models, then marks both datasets consumed **before opening test inputs**.
You will be prompted for:

5. `datasets\spendly_synthetic_r3\HOLDOUT_DO_NOT_TUNE\test.zip`
6. `datasets\spendly_synthetic_r3\HOLDOUT_DO_NOT_TUNE\test_shifted.zip`

**After the final-test cell finishes, run the next cell: download-final-and-handoff.**
It rebuilds and downloads the final ZIP containing both test reports. The earlier
development ZIP does not contain those new results.

No fitting, selection, threshold adjustment or scaler update occurs on either file.
Both test user groups must be distinct from development users. Their first 120 weeks
are causal warmup only; evaluation uses later weeks. Every selected comparator stays
visible. Changing MODE or run folder does not make viewed test data fresh. The ledger
lives in WORK_DIR; preserve it across Kaggle sessions. A reset/local ledger is not a
global registry. Failed attempts are conservatively consumed too. Save the reports.


In [ ]:
FINAL_CONFIRMATION = ''  # set exactly: EVALUATE BOTH FROZEN R3 HOLDOUTS ONCE
if FINAL_CONFIRMATION == 'EVALUATE BOTH FROZEN R3 HOLDOUTS ONCE':
    if MODE != 'full':
        raise ValueError('Do not consume final holdouts for a smoke run')
    FROZEN=load_r3_frozen(RUN_DIR,RUN_ID)
    if FROZEN['manifest']['settings']['mode'] != 'full':
        raise ValueError('Frozen models belong to smoke mode; do not evaluate real holdouts')
    if FROZEN['manifest']['settings'] != SETTINGS:
        raise ValueError('Settings changed after freezing; restore the frozen configuration')
    # Restore verified development ID sets without changing any model.
    development_ids=set()
    for name in ('train','validation','calibration'):
        if file_hash(COHORT_FILES[name]) != DATASET['cohorts'][name]['sha256']:
            raise ValueError('Development input file changed after verification')
        for uid,_ in grouped_csv(COHORT_FILES[name]):
            development_ids.add(uid)
    # Discover BOTH paths before consumption, without opening test contents.
    FINAL_PATHS={name:request_one_file('Locate holdout '+name,name+'.csv',COHORT_PATHS[name])
                 for name in ('test','test_shifted')}
    v6_final_claim(DATASET,HOLDOUT_STATE_DIR,RUN_ID,[WORK_DIR,*PREVIOUS_STATE_DIRS])
    final_ids=set()
    for number,name in [(5,'test'),(6,'test_shifted')]:
        try:
            uploaded=FINAL_PATHS[name]
            path=verify_cohort_file(uploaded,name,DATASET,WORK_DIR/'verified_csv')
            test_data=cached_features(path,name,DATASET,WORK_DIR/'feature_cache',SETTINGS)
            if test_data['users'] & (development_ids|final_ids):
                raise ValueError('Test users overlap another cohort')
            final_ids |= test_data['users']
            result=evaluate_r3(FROZEN,test_data,SETTINGS,name)
            json_write(RUN_DIR/(name+'_result.json'),result)
            for kind,rows in result.items():
                pd.DataFrame(rows).to_csv(RUN_DIR/(name+'_'+kind+'.csv'),index=False)
            print(name,'FORECAST:',pd.DataFrame(result['forecast']).round(4).to_string(index=False))
            print(name,'ANOMALY:',pd.DataFrame(result['anomaly']).round(4).to_string(index=False))
            del test_data
        except Exception as error:
            json_write(RUN_DIR/(name+'_failure.json'),dict(status='consumed_failed',error_type=type(error).__name__))
            archive_results(RUN_DIR)
            raise
else:
    print('FINAL TEST NOT RUN. Development results can be reviewed without consuming holdouts.')


In [ ]:
handoff=dict(protocol='v6-r3-kaggle-v1',run_id=RUN_ID,mode=MODE,
    forecast_validation=pd.read_csv(RUN_DIR/'forecast_validation.csv').to_dict('records'),
    anomaly_validation=pd.read_csv(RUN_DIR/'anomaly_validation.csv').to_dict('records'),
    calibration=pd.read_csv(RUN_DIR/'anomaly_calibration.csv').to_dict('records'),
    final_results={name:json.loads((RUN_DIR/(name+'_result.json')).read_text()) for name in ('test','test_shifted') if (RUN_DIR/(name+'_result.json')).exists()},
    errors={name:json.loads((RUN_DIR/(name+'_failure.json')).read_text()) for name in ('test','test_shifted') if (RUN_DIR/(name+'_failure.json')).exists()},
    limitations=['synthetic data only','candidate artifacts not yet API-compatible','device and real-user validation untested'])
json_write(RUN_DIR/'handoff.json',handoff)
print('SPENDLY_R3_RESULT_START')
print(json.dumps(json_safe(handoff),indent=2,allow_nan=False))
print('SPENDLY_R3_RESULT_END')
archive=archive_results(RUN_DIR)
print('SEND BACK:',archive.name,'and optionally File > Download Notebook')
print('Download this existing ZIP from the working-file/output browser NOW, and preserve holdout_ledger.')
print('After final evaluation DO NOT Save & Run All: that executes a new experiment. Use non-executing save only.')
FINAL_CONFIRMATION = ''  # runtime guard; also clear the literal in the final-test source before any future Run All
